<a href="https://colab.research.google.com/github/ImanMahato05/EcoHydro_Project/blob/main/Ecohydro_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls -la /content/drive

total 12
drwxr-xr-x 3 root root 4096 Sep 26 04:08 .
drwxr-xr-x 1 root root 4096 Sep 26 04:08 ..
drwxr-xr-x 3 root root 4096 Sep 26 04:08 MyDrive


In [ ]:
!rm -rf /content/drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q earthengine-api==1.7.31 geemap==0.38.3
!pip install -q xarray==2026.4.0 netCDF4==1.7.4 rioxarray
!pip install -q geopandas==1.1.3 rasterstats==0.21.0 shapely pyproj
!pip install -q dataretrieval==1.2.0 metloom==0.9.4 "cdsapi>=0.7.7"
!pip install -q statsmodels linearmodels scikit-learn dowhy==0.14

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.4/480.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.5/342.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import ee
ee.Authenticate()
ee.Initialize(project='ecohydro-project')
print(ee.String('ok').getInfo())
print("EE:", ee.String('ok').getInfo())

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


ok
EE: ok


In [3]:
from google.colab import userdata
import os

netrc_content = f"machine urs.earthdata.nasa.gov\nlogin {userdata.get('EARTHDATA_USER')}\npassword {userdata.get('EARTHDATA_PASS')}\n"
with open('/root/.netrc', 'w') as f:
    f.write(netrc_content)
os.chmod('/root/.netrc', 0o600)

cds_content = f"url: https://cds.climate.copernicus.eu/api\nkey: {userdata.get('CDS_API_KEY')}\n"
with open('/root/.cdsapirc', 'w') as f:
    f.write(cds_content)

usgs_key = userdata.get('USGS_API_KEY')
print("Earthdata and CDS config written. USGS key loaded into memory.")

Earthdata and CDS config written. USGS key loaded into memory.


In [ ]:
# %% Step 1: Load and QA the master well metadata
import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/ecohydro_project'
wells_path = f'{base}/data/raw/wells_metadata/usgs_wells_conus_metadata.csv'

wells = pd.read_csv(wells_path, dtype={'monitoring_location_id': str})
print(f"Loaded {len(wells):,} well records")

# Standardize ID as string
wells['monitoring_location_id'] = wells['monitoring_location_id'].astype(str).str.strip()

# Check uniqueness
n_dupes = wells['monitoring_location_id'].duplicated().sum()
print(f"Duplicate monitoring_location_id: {n_dupes}")
if n_dupes > 0:
    wells = wells.drop_duplicates(subset=['monitoring_location_id'], keep='first')
    print(f"Dropped duplicates -> {len(wells):,} remaining")

# Check site_type == Well
if 'site_type' in wells.columns:
    non_well = (wells['site_type'] != 'Well').sum()
    print(f"Rows with site_type != 'Well': {non_well:,}")
    wells = wells[wells['site_type'] == 'Well'].copy()

# Coordinate validity
valid_coords = (
    wells['latitude'].notna() & wells['longitude'].notna() &
    wells['latitude'].between(-90, 90) & wells['longitude'].between(-180, 180)
)
print(f"Wells with valid coordinates: {valid_coords.sum():,} / {len(wells):,}")
wells = wells[valid_coords].copy()

# HUC availability (for later QA cross-check only)
huc_missing = wells['hydrologic_unit_code'].isna().sum()
print(f"Wells missing hydrologic_unit_code: {huc_missing:,} / {len(wells):,}")

print(f"\nFinal wells after Step 1 QA: {len(wells):,}")

Loaded 1,223,831 well records
Duplicate monitoring_location_id: 0
Rows with site_type != 'Well': 0
Wells with valid coordinates: 1,223,819 / 1,223,831
Wells missing hydrologic_unit_code: 15,982 / 1,223,819

Final wells after Step 1 QA: 1,223,819


In [ ]:
# %% Step 2: Spatial assignment to HUC02 basins (authoritative), HUC prefix as QA cross-check only
import geopandas as gpd
from shapely.geometry import Point

huc02_path = f'{base}/outputs/vectors/domestic_basins_huc02.geojson'
huc02 = gpd.read_file(huc02_path)

# Standardize HUC02 identifier in the polygon layer
huc02_id_col = 'huc2' if 'huc2' in huc02.columns else [c for c in huc02.columns if 'huc' in c.lower()][0]
huc02[huc02_id_col] = huc02[huc02_id_col].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(2)

expected_huc2s = {'01','02','03','04','05','07','08','09','10','11','12','13','14','15','16','17','18'}
present = set(huc02[huc02_id_col].unique())
print(f"HUC02 polygons present: {sorted(present)}")
if present != expected_huc2s:
    print(f"WARNING: polygon set differs from expected 17-basin set. Missing: {expected_huc2s - present}, Extra: {present - expected_huc2s}")

# Build well points GeoDataFrame (assume WGS84 lat/lon, EPSG:4326)
wells_gdf = gpd.GeoDataFrame(
    wells.copy(),
    geometry=[Point(xy) for xy in zip(wells['longitude'], wells['latitude'])],
    crs='EPSG:4326'
)

# Reproject wells to match HUC02 polygon CRS if needed
if huc02.crs != wells_gdf.crs:
    wells_gdf = wells_gdf.to_crs(huc02.crs)

# Spatial join: assign each well to the HUC02 polygon it falls inside
wells_joined = gpd.sjoin(wells_gdf, huc02[[huc02_id_col, 'geometry']], how='left', predicate='within')
wells_joined = wells_joined.rename(columns={huc02_id_col: 'huc2_spatial'})

n_assigned = wells_joined['huc2_spatial'].notna().sum()
print(f"Wells spatially assigned to a HUC02 polygon: {n_assigned:,} / {len(wells_joined):,}")

# QA cross-check: compare against first two digits of hydrologic_unit_code
# UPDATED: adaptive padding to handle 8–12 digit HUC codes correctly
def extract_huc2_prefix(huc_value, pad_width=12):
    if pd.isna(huc_value):
        return pd.NA
    huc_str = str(int(float(huc_value))).zfill(pad_width)
    return huc_str[:2]

wells_joined['huc2_from_code'] = wells_joined['hydrologic_unit_code'].apply(extract_huc2_prefix)

mismatch_mask = (
    wells_joined['huc2_spatial'].notna() &
    wells_joined['huc2_from_code'].notna() &
    (wells_joined['huc2_spatial'] != wells_joined['huc2_from_code'])
)
n_mismatch = mismatch_mask.sum()
print(f"QA cross-check mismatches (spatial vs HUC-code prefix): {n_mismatch:,} "
      f"({100*n_mismatch/max(wells_joined['huc2_spatial'].notna().sum(),1):.2f}% of assigned wells)")

# NEW: flag residual mismatches for later inspection (do NOT use this to reassign basins)
wells_joined['huc2_mismatch_residual'] = mismatch_mask
n_residual = wells_joined['huc2_mismatch_residual'].sum()
print(f"Residual mismatches retained for later inspection: {n_residual:,} "
      f"({100*n_residual/wells_joined['huc2_spatial'].notna().sum():.2f}%)")

# Drop the geopandas index_right helper column if present
if 'index_right' in wells_joined.columns:
    wells_joined = wells_joined.drop(columns=['index_right'])

HUC02 polygons present: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']
Wells spatially assigned to a HUC02 polygon: 1,223,779 / 1,223,819
QA cross-check mismatches (spatial vs HUC-code prefix): 103,041 (8.42% of assigned wells)
Residual mismatches retained for later inspection: 103,041 (8.42%)


In [ ]:
# %% Step 3: Restrict to 17 HUC02 basins, build basin-level metadata table
wells_17 = wells_joined[wells_joined['huc2_spatial'].isin(expected_huc2s)].copy()
wells_17 = wells_17.rename(columns={'huc2_spatial': 'huc2'})
print(f"Wells within the 17 HUC02 basins: {len(wells_17):,} / {len(wells_joined):,}")

basin_area_path = f'{base}/data/processed/basin_selection/huc2_master_table_with_dem_v3_and_densities.csv'
basin_master = pd.read_csv(basin_area_path)
basin_master['huc2'] = basin_master['huc2'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(2)

well_counts = wells_17.groupby('huc2').agg(
    total_usgs_wells=('monitoring_location_id', 'count'),
    wells_valid_coords=('monitoring_location_id', 'count')  # already filtered to valid coords in Step 1
).reset_index()

basin_wells_table = basin_master[['huc2', 'name', 'area_km2_computed']].merge(
    well_counts, on='huc2', how='left'
).fillna({'total_usgs_wells': 0, 'wells_valid_coords': 0})

basin_wells_table['well_density_per_100k_km2'] = (
    basin_wells_table['total_usgs_wells'] / basin_wells_table['area_km2_computed'] * 100000
)

print(basin_wells_table.to_string(index=False))

# Save basin-level metadata and the restricted well list (with monitoring_location_id retained)
out_dir = f'{base}/data/processed/basin_selection'
basin_wells_table.to_csv(f'{out_dir}/huc2_well_metadata_summary.csv', index=False)
wells_17.to_csv(f'{out_dir}/usgs_wells_huc02_restricted_metadata.csv', index=False)
print(f"\nSaved basin-level well metadata and restricted well list to {out_dir}")

Wells within the 17 HUC02 basins: 1,216,921 / 1,223,819
huc2                       name  area_km2_computed  total_usgs_wells  wells_valid_coords  well_density_per_100k_km2
  01         New England Region       1.988462e+05             85159               85159               42826.559917
  02        Mid Atlantic Region       2.764838e+05            126587              126587               45784.592090
  03 South Atlantic-Gulf Region       7.396592e+05            131898              131898               17832.265831
  04         Great Lakes Region       6.170527e+05             44740               44740                7250.596793
  05                Ohio Region       4.219575e+05             44501               44501               10546.323736
  07   Upper Mississippi Region       4.920325e+05             51479               51479               10462.520171
  08   Lower Mississippi Region       2.759616e+05             81322               81322               29468.593098
  09    Souris-R

In [ ]:
# %% Step 4a: Prepare full well population for retrieval (no pre-filtering)
import os

gw_levels_dir = f'{base}/data/processed/basin_selection/groundwater_levels'
os.makedirs(gw_levels_dir, exist_ok=True)

# Use ALL wells in the 17-basin population
candidate_well_ids = wells_17['monitoring_location_id'].tolist()
print(f"Proceeding with FULL population: {len(candidate_well_ids):,} wells")
print("This will take ~3-4 days at ~3-4 wells/sec. Step 4b is checkpointed, so you can resume across sessions.")

# Save the manifest of all well IDs to be processed
manifest_path = f'{gw_levels_dir}/well_manifest_full_population.csv'
pd.DataFrame({'monitoring_location_id': candidate_well_ids}).to_csv(manifest_path, index=False)
print(f"Saved well manifest: {manifest_path}")

Proceeding with FULL population: 1,216,921 wells
This will take ~3-4 days at ~3-4 wells/sec. Step 4b is checkpointed, so you can resume across sessions.
Saved well manifest: /content/drive/MyDrive/ecohydro_project/data/processed/basin_selection/groundwater_levels/well_manifest_full_population.csv


In [1]:
# # %% Step 4b: Batched groundwater-level retrieval with multi-ID requests and checkpointing
# from dataretrieval import waterdata
# import time
# from datetime import datetime
# import pandas as pd
# import os

# # Configuration
# GW_PARAM_CODES = ['62610', '62611', '72019']  # 62610=GW level above NGVD29, 62611=GW level above NAVD88, 72019=depth to water below land surface
# BATCH_SIZE_WELLS = 20  # Start conservative; can increase after benchmarking
# gw_levels_dir = f'{base}/data/processed/basin_selection/groundwater_levels'
# batch_dir = f'{gw_levels_dir}/batches_full_population'
# os.makedirs(batch_dir, exist_ok=True)

# # Load the manifest
# manifest_path = f'{gw_levels_dir}/well_manifest_full_population.csv'
# manifest_df = pd.read_csv(manifest_path, dtype={'monitoring_location_id': str})
# candidate_well_ids = manifest_df['monitoring_location_id'].tolist()

# # Checkpoint log: tracks which batches are complete
# retrieval_log_path = f'{gw_levels_dir}/retrieval_checkpoint_log_full.csv'

# def already_processed_batches():
#     """Read checkpoint log and return set of completed batch numbers."""
#     if os.path.exists(retrieval_log_path):
#         done = pd.read_csv(retrieval_log_path)
#         return set(done['batch_num'].tolist())
#     return set()

# completed_batches = already_processed_batches()
# print(f"Batches already completed: {len(completed_batches)}")
# print(f"Starting retrieval for {len(candidate_well_ids):,} wells in batches of {BATCH_SIZE_WELLS} wells each")

# start_time = time.time()
# retrieval_log = []
# n_total_batches = (len(candidate_well_ids) + BATCH_SIZE_WELLS - 1) // BATCH_SIZE_WELLS

# for batch_num, start in enumerate(range(0, len(candidate_well_ids), BATCH_SIZE_WELLS)):
#     if batch_num in completed_batches:
#         continue  # Skip already-completed batches

#     batch_ids = candidate_well_ids[start:start + BATCH_SIZE_WELLS]
#     batch_records = []
#     batch_errors = []
#     n_success, n_no_data, n_error = 0, 0, 0

#     try:
#         # Multi-ID request: retrieve groundwater levels for all wells in this batch at once
#         df, md = waterdata.get_field_measurements(monitoring_location_id=batch_ids)

#         if len(df) > 0:
#             # Filter to groundwater-level parameter codes
#             gw_df = df[df['parameter_code'].isin(GW_PARAM_CODES)].copy()

#             if len(gw_df) > 0:
#                 # Ensure monitoring_location_id column exists
#                 if 'monitoring_location_id' not in gw_df.columns:
#                     # If multi-ID request didn't include this column, we need to infer from context
#                     # For now, assume it's present; if not, we'll need to split into single-well requests
#                     print(f"  WARNING: monitoring_location_id missing from response; may need to fall back to single-well requests")

#                 batch_records.append(gw_df)
#                 n_success = len(gw_df['monitoring_location_id'].unique()) if 'monitoring_location_id' in gw_df.columns else len(batch_ids)
#             else:
#                 n_no_data = len(batch_ids)  # No GW-level measurements in this batch
#         else:
#             n_no_data = len(batch_ids)  # No data at all for these wells

#     except Exception as e:
#         # Log error for entire batch; could retry with smaller batches or single-well requests
#         error_msg = f"{type(e).__name__}: {e}"
#         batch_errors.append({'batch_wells': ','.join(batch_ids), 'error': error_msg})
#         n_error = len(batch_ids)
#         print(f"  Batch {batch_num} failed: {error_msg}")

#     # Write this batch to its own file immediately (never re-concat the whole accumulated set)
#     if batch_records:
#         batch_df = pd.concat(batch_records, ignore_index=True)
#         batch_df.to_csv(f'{batch_dir}/batch_{batch_num:05d}.csv', index=False)

#     if batch_errors:
#         pd.DataFrame(batch_errors).to_csv(f'{batch_dir}/batch_{batch_num:05d}_errors.csv', index=False)

#     # Update checkpoint log
#     retrieval_log.append({
#         'batch_num': batch_num,
#         'n_wells_attempted': len(batch_ids),
#         'n_success': n_success,
#         'n_no_data': n_no_data,
#         'n_errors': n_error,
#         'timestamp': datetime.now().isoformat()
#     })
#     pd.DataFrame(retrieval_log).to_csv(
#         retrieval_log_path, index=False,
#         mode='a' if os.path.exists(retrieval_log_path) else 'w',
#         header=not os.path.exists(retrieval_log_path)
#     )

#     # Progress report every 10 batches
#     if batch_num % 10 == 0:
#         elapsed = time.time() - start_time
#         n_done = start + len(batch_ids)
#         rate = n_done / elapsed if elapsed > 0 else 0
#         eta_min = (len(candidate_well_ids) - n_done) / rate / 60 if rate > 0 else 0
#         eta_days = eta_min / 1440 if eta_min > 0 else 0
#         print(f"[Batch {batch_num}/{n_total_batches}] {n_done:,}/{len(candidate_well_ids):,} wells "
#               f"({rate:.1f} wells/sec, ETA: {eta_min:.0f} min = {eta_days:.1f} days)")

# print(f"\nRetrieval complete for {len(candidate_well_ids):,} wells.")
# total_time = time.time() - start_time
# print(f"Total runtime: {total_time/3600:.1f} hours = {total_time/86400:.1f} days")

In [2]:
# # %% Step 4b (REVISED): Batched retrieval with exponential backoff on 429 errors
# from dataretrieval import waterdata
# import time
# from datetime import datetime
# import pandas as pd
# import os

# GW_PARAM_CODES = ['62610', '62611', '72019']
# BATCH_SIZE_WELLS = 10  # Reduced from 20 to stay under rate limits
# gw_levels_dir = f'{base}/data/processed/basin_selection/groundwater_levels'
# batch_dir = f'{gw_levels_dir}/batches_full_population'
# os.makedirs(batch_dir, exist_ok=True)

# manifest_path = f'{gw_levels_dir}/well_manifest_full_population.csv'
# manifest_df = pd.read_csv(manifest_path, dtype={'monitoring_location_id': str})
# candidate_well_ids = manifest_df['monitoring_location_id'].tolist()

# retrieval_log_path = f'{gw_levels_dir}/retrieval_checkpoint_log_full.csv'

# def already_processed_batches():
#     if os.path.exists(retrieval_log_path):
#         done = pd.read_csv(retrieval_log_path)
#         return set(done['batch_num'].tolist())
#     return set()

# completed_batches = already_processed_batches()
# print(f"Batches already completed: {len(completed_batches)}")
# print(f"Starting retrieval for {len(candidate_well_ids):,} wells in batches of {BATCH_SIZE_WELLS} wells each")

# start_time = time.time()
# retrieval_log = []
# n_total_batches = (len(candidate_well_ids) + BATCH_SIZE_WELLS - 1) // BATCH_SIZE_WELLS

# MAX_RETRIES = 5
# BASE_DELAY_SEC = 30  # Start with 30-second delay on 429

# for batch_num, start in enumerate(range(0, len(candidate_well_ids), BATCH_SIZE_WELLS)):
#     if batch_num in completed_batches:
#         continue

#     batch_ids = candidate_well_ids[start:start + BATCH_SIZE_WELLS]
#     batch_records = []
#     batch_errors = []
#     n_success, n_no_data, n_error = 0, 0, 0

#     # Retry loop with exponential backoff for 429 errors
#     for attempt in range(MAX_RETRIES):
#         try:
#             df, md = waterdata.get_field_measurements(monitoring_location_id=batch_ids)

#             if len(df) > 0:
#                 gw_df = df[df['parameter_code'].isin(GW_PARAM_CODES)].copy()
#                 if len(gw_df) > 0:
#                     batch_records.append(gw_df)
#                     n_success = len(gw_df['monitoring_location_id'].unique()) if 'monitoring_location_id' in gw_df.columns else len(batch_ids)
#                 else:
#                     n_no_data = len(batch_ids)
#             else:
#                 n_no_data = len(batch_ids)

#             # Success - break out of retry loop
#             break

#         except Exception as e:
#             error_msg = f"{type(e).__name__}: {e}"

#             # Check if this is a 429 rate-limit error
#             if '429' in error_msg or 'RateLimited' in error_msg or 'QuotaExhausted' in error_msg:
#                 if attempt < MAX_RETRIES - 1:
#                     delay = BASE_DELAY_SEC * (2 ** attempt)  # Exponential backoff
#                     print(f"  Batch {batch_num} hit rate limit (attempt {attempt+1}/{MAX_RETRIES}). Waiting {delay} seconds before retry...")
#                     time.sleep(delay)
#                 else:
#                     # Max retries exceeded - log as error and move on
#                     batch_errors.append({'batch_wells': ','.join(batch_ids), 'error': f"Rate limit exceeded after {MAX_RETRIES} retries: {error_msg}"})
#                     n_error = len(batch_ids)
#                     print(f"  Batch {batch_num} FAILED permanently after {MAX_RETRIES} retries.")
#             else:
#                 # Non-rate-limit error - log and move on without retry
#                 batch_errors.append({'batch_wells': ','.join(batch_ids), 'error': error_msg})
#                 n_error = len(batch_ids)
#                 print(f"  Batch {batch_num} failed with non-retryable error: {error_msg}")
#                 break

#     # Write batch outputs
#     if batch_records:
#         batch_df = pd.concat(batch_records, ignore_index=True)
#         batch_df.to_csv(f'{batch_dir}/batch_{batch_num:05d}.csv', index=False)

#     if batch_errors:
#         pd.DataFrame(batch_errors).to_csv(f'{batch_dir}/batch_{batch_num:05d}_errors.csv', index=False)

#     # Update checkpoint log
#     retrieval_log.append({
#         'batch_num': batch_num,
#         'n_wells_attempted': len(batch_ids),
#         'n_success': n_success,
#         'n_no_data': n_no_data,
#         'n_errors': n_error,
#         'timestamp': datetime.now().isoformat()
#     })
#     pd.DataFrame(retrieval_log).to_csv(
#         retrieval_log_path, index=False,
#         mode='a' if os.path.exists(retrieval_log_path) else 'w',
#         header=not os.path.exists(retrieval_log_path)
#     )

#     # Progress report
#     if batch_num % 10 == 0:
#         elapsed = time.time() - start_time
#         n_done = start + len(batch_ids)
#         rate = n_done / elapsed if elapsed > 0 else 0
#         eta_min = (len(candidate_well_ids) - n_done) / rate / 60 if rate > 0 else 0
#         print(f"[Batch {batch_num}/{n_total_batches}] {n_done:,}/{len(candidate_well_ids):,} wells "
#               f"({rate:.1f} wells/sec, ETA: {eta_min:.0f} min)")

# print(f"\nRetrieval complete for {len(candidate_well_ids):,} wells.")

In [ ]:
# %% Step 4c: Combine all batch files ONCE at the end
import glob

batch_files = sorted(glob.glob(f'{batch_dir}/batch_*.csv'))
batch_files = [f for f in batch_files if 'errors' not in f]

if batch_files:
    gw_levels_df = pd.concat([pd.read_csv(f) for f in batch_files], ignore_index=True)
    gw_levels_df.to_csv(f'{gw_levels_dir}/huc02_wells_groundwater_levels_raw.csv', index=False)
    print(f"Combined {len(batch_files):,} batch files -> {len(gw_levels_df):,} total observation rows")
else:
    gw_levels_df = pd.DataFrame()
    print("No batch files found to combine.")

# Combine error logs
error_files = sorted(glob.glob(f'{batch_dir}/batch_*_errors.csv'))
if error_files:
    all_errors = pd.concat([pd.read_csv(f) for f in error_files], ignore_index=True)
    all_errors.to_csv(f'{gw_levels_dir}/all_retrieval_errors.csv', index=False)
    print(f"Combined error log: {len(all_errors):,} errors")

In [ ]:
# %% Step 5: Deduplicate observations by (well, date), with EXPLICIT date parsing
import pandas as pd
import numpy as np

gw_levels_dir = f'{base}/data/processed/basin_selection/groundwater_levels'
gw_levels_df = pd.read_csv(f'{gw_levels_dir}/huc02_wells_groundwater_levels_raw.csv')

def parse_explicit_date(row):
    """Parse date from year/month/day columns explicitly, avoiding pd.to_datetime inference warnings."""
    try:
        return pd.Timestamp(year=int(row['year']), month=int(row['month']), day=int(row['day']))
    except (ValueError, TypeError, KeyError):
        return pd.NaT

gw_levels_df['measurement_dt'] = gw_levels_df.apply(parse_explicit_date, axis=1)

n_unparsed = gw_levels_df['measurement_dt'].isna().sum()
print(f"Rows with unparseable dates: {n_unparsed:,} / {len(gw_levels_df):,}")
if n_unparsed > 0:
    gw_levels_df.loc[gw_levels_df['measurement_dt'].isna()].to_csv(
        f'{gw_levels_dir}/unparseable_date_rows.csv', index=False
    )

gw_levels_df = gw_levels_df[gw_levels_df['measurement_dt'].notna()].copy()
gw_levels_df['obs_date'] = gw_levels_df['measurement_dt'].dt.date

# Count distinct (well, date) events and how many have multiple parameter codes on the same date
event_counts = gw_levels_df.groupby(['monitoring_location_id', 'obs_date']).size().reset_index(name='n_param_codes_this_date')
print(f"Distinct (well, date) observation events: {len(event_counts):,}")
print(f"Events with more than one parameter code on the same date: {(event_counts['n_param_codes_this_date'] > 1).sum():,}")

In [ ]:
# %% Step 6: Per-well temporal metrics + completeness metric
def compute_well_temporal_metrics(group):
    """Calculate temporal metrics for a single well's observation dates."""
    dates = sorted(pd.to_datetime(group['obs_date'].unique()))

    if len(dates) == 0:
        return pd.Series({
            'first_gw_date': pd.NaT, 'last_gw_date': pd.NaT,
            'record_years': np.nan, 'n_observation_dates': 0,
            'n_observation_years': 0, 'max_gap_years': np.nan,
            'year_coverage_ratio': np.nan
        })

    first_date, last_date = dates[0], dates[-1]
    record_years = (last_date - first_date).days / 365.25
    n_obs_dates = len(dates)
    obs_years = set(d.year for d in dates)
    n_obs_years = len(obs_years)

    # Corrected year_coverage_ratio: bounded between 0 and 1
    span_years = last_date.year - first_date.year + 1
    year_coverage_ratio = n_obs_years / span_years if span_years > 0 else np.nan

    if len(dates) > 1:
        gaps = [(dates[i+1] - dates[i]).days / 365.25 for i in range(len(dates)-1)]
        max_gap = max(gaps)
    else:
        max_gap = np.nan

    return pd.Series({
        'first_gw_date': first_date, 'last_gw_date': last_date,
        'record_years': record_years, 'n_observation_dates': n_obs_dates,
        'n_observation_years': n_obs_years, 'max_gap_years': max_gap,
        'year_coverage_ratio': year_coverage_ratio
    })

well_temporal = gw_levels_df.groupby('monitoring_location_id').apply(
    compute_well_temporal_metrics, include_groups=False
).reset_index()

print(well_temporal.describe(include='all'))

In [ ]:
# %% Step 7: Documented project criterion for "long monitoring-well record"
# Criterion (project-defined, not a USGS standard):
#   record_years >= 10  AND  n_observation_years >= 5

LONG_RECORD_MIN_YEARS = 10
LONG_RECORD_MIN_DISTINCT_YEARS = 5

well_temporal['long_record_flag'] = (
    (well_temporal['record_years'] >= LONG_RECORD_MIN_YEARS) &
    (well_temporal['n_observation_years'] >= LONG_RECORD_MIN_DISTINCT_YEARS)
)

print(f"Wells meeting long-record criterion: {well_temporal['long_record_flag'].sum():,} / {len(well_temporal):,}")

In [ ]:
# %% Step 8: Well-level summary file
wells_17_path = f'{base}/data/processed/basin_selection/usgs_wells_huc02_restricted_metadata.csv'
wells_17 = pd.read_csv(wells_17_path, dtype={'monitoring_location_id': str})

well_summary = wells_17[[
    'monitoring_location_id', 'huc2', 'latitude', 'longitude',
    'state_name', 'county_name', 'aquifer_code', 'national_aquifer_code',
    'well_constructed_depth', 'hole_constructed_depth', 'hydrologic_unit_code'
]].merge(well_temporal, on='monitoring_location_id', how='left')

# Fill NaN long_record_flag for wells with no groundwater observations
well_summary['long_record_flag'] = well_summary['long_record_flag'].fillna(False)

well_summary_path = f'{base}/data/processed/basin_selection/usgs_wells_huc02_groundwater_record_summary.csv'
well_summary.to_csv(well_summary_path, index=False)
print(f"Saved well-level summary: {well_summary_path} ({len(well_summary):,} rows)")

In [ ]:
# %% Step 9: Aggregate to HUC02-basin level (keep all 17 basins, even with zero counts)
basin_master_path = f'{base}/data/processed/basin_selection/huc2_master_table_with_dem_v3_and_densities.csv'
basin_master = pd.read_csv(basin_master_path)
basin_master['huc2'] = basin_master['huc2'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(2)

basin_long_record = well_summary.groupby('huc2').agg(
    total_wells=('monitoring_location_id', 'count'),
    wells_with_gw_obs=('n_observation_dates', lambda x: (x.fillna(0) > 0).sum()),
    long_record_wells=('long_record_flag', 'sum')
).reset_index()

# Left-merge on the FULL 17-basin list so every basin appears, even with zero wells
basin_long_record = basin_master[['huc2', 'name', 'area_km2_computed']].merge(
    basin_long_record, on='huc2', how='left'
)
basin_long_record[['total_wells', 'wells_with_gw_obs', 'long_record_wells']] = (
    basin_long_record[['total_wells', 'wells_with_gw_obs', 'long_record_wells']].fillna(0)
)
basin_long_record['long_record_well_density_per_100k_km2'] = (
    basin_long_record['long_record_wells'] / basin_long_record['area_km2_computed'] * 100000
)

out_path = f'{base}/data/processed/basin_selection/huc2_long_monitoring_well_counts.csv'
basin_long_record.to_csv(out_path, index=False)
print(f"Saved basin-level long-record well counts: {out_path}")
print(basin_long_record.to_string(index=False))

In [ ]:
# %% Step 10: Expanded spot-check across multiple HUC02 basins
import numpy as np

np.random.seed(42)
sample_basins = well_summary['huc2'].dropna().unique()

print("Spot-checking wells from multiple HUC02 basins...\n")

for huc2 in sample_basins[:5]:  # Check a handful of basins
    subset = well_summary[well_summary['huc2'] == huc2]
    long_sub = subset[subset['long_record_flag']]
    short_sub = subset[~subset['long_record_flag']]

    print(f"=== HUC02 {huc2} ===")

    if len(long_sub) > 0:
        s = long_sub.sample(min(2, len(long_sub)), random_state=1)
        print("Long-record sample:")
        print(s[['monitoring_location_id','first_gw_date','last_gw_date','record_years',
                  'n_observation_years','max_gap_years','year_coverage_ratio']].to_string(index=False))

        # Verify against raw observations for one well
        check_id = s['monitoring_location_id'].iloc[0]
        raw_check = gw_levels_df[gw_levels_df['monitoring_location_id'] == check_id]
        print(f"  Raw observation dates for {check_id}: {sorted(raw_check['obs_date'].unique())[:5]} ... "
              f"({raw_check['obs_date'].nunique()} distinct dates, {len(raw_check)} raw rows)")
    else:
        print("  No long-record wells in this basin")

    if len(short_sub) > 0:
        s = short_sub.sample(min(2, len(short_sub)), random_state=1)
        print("Short-record sample:")
        print(s[['monitoring_location_id','first_gw_date','last_gw_date','record_years',
                  'n_observation_years','max_gap_years']].to_string(index=False))
    else:
        print("  No short-record wells in this basin")

    print()

print("Only after these checks pass should huc2_long_monitoring_well_counts.csv be merged "
      "into the Stage 2.3 ground-constraint table alongside GPS, SNOTEL, and stream-gauge counts.")

###SNOTEL DOWNLOAD --------------------------------------------------------------------------------------

In [ ]:
# %% Step 1: Download SNOTEL station metadata (active + inactive) and save to Drive
import requests
import pandas as pd
import os
from datetime import datetime

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'
os.makedirs(snotel_dir, exist_ok=True)

# AWDB REST API endpoint for stations
url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"

# Download ACTIVE SNOTEL stations
print("Downloading ACTIVE SNOTEL stations...")
params_active = {
    "stationTriplets": "*:*:SNTL",
    "activeOnly": "true",
    "returnForecastPointMetadata": "false",
    "returnReservoirMetadata": "false",
    "returnStationElements": "false"
}

response = requests.get(url, params=params_active, timeout=60)
response.raise_for_status()
data_active = response.json()

print(f"Status code: {response.status_code}")
print(f"Number of ACTIVE SNOTEL stations returned: {len(data_active)}")

# Download INACTIVE SNOTEL stations (if available)
print("\nDownloading INACTIVE SNOTEL stations...")
params_inactive = {
    "stationTriplets": "*:*:SNTL",
    "activeOnly": "false",
    "returnForecastPointMetadata": "false",
    "returnReservoirMetadata": "false",
    "returnStationElements": "false"
}

response = requests.get(url, params=params_inactive, timeout=60)
response.raise_for_status()
data_inactive = response.json()

print(f"Status code: {response.status_code}")
print(f"Number of INACTIVE SNOTEL stations returned: {len(data_inactive)}")

# Combine active and inactive stations
df_active = pd.DataFrame(data_active)
df_inactive = pd.DataFrame(data_inactive)

# Add active flag
df_active['active_flag'] = True
df_inactive['active_flag'] = False

# Combine
df_all = pd.concat([df_active, df_inactive], ignore_index=True)

# Remove duplicates by stationTriplet (keep first occurrence)
n_dupes = df_all['stationTriplet'].duplicated().sum()
print(f"\nDuplicate stationTriplets found: {n_dupes}")
if n_dupes > 0:
    df_all = df_all.drop_duplicates(subset=['stationTriplet'], keep='first')
    print(f"After removing duplicates: {len(df_all):,} unique stations")

print(f"\nTotal unique SNOTEL stations (active + inactive): {len(df_all):,}")
print(f"  - Active: {df_all['active_flag'].sum():,}")
print(f"  - Inactive: {(~df_all['active_flag']).sum():,}")

# Save RAW station metadata
raw_file = f'{snotel_dir}/nrcs_snotel_stations_raw.csv'
df_all.to_csv(raw_file, index=False)
print(f"\nSaved raw station metadata: {raw_file}")

# Standardize column names to match wells metadata format
rename_map = {
    'stationTriplet': 'station_triplet',
    'stationId': 'station_id',
    'stateCode': 'state_code',
    'networkCode': 'network_code',
    'name': 'station_name',
    'dcoCode': 'dco_code',
    'countyName': 'county_name',
    'huc': 'hydrologic_unit_code',
    'elevation': 'altitude',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'dataTimeZone': 'time_zone',
    'shefId': 'shef_id',
    'operator': 'operator',
    'beginDate': 'start_date',
    'endDate': 'end_date',
    'associatedHucs': 'associated_hucs',
    'pedonCode': 'pedon_code'
}

df_std = df_all.rename(columns=rename_map).copy()

# Extract HUC-8 from hydrologic_unit_code (first 8 digits)
df_std['huc8'] = df_std['hydrologic_unit_code'].str[:8]

# Add a unique ID column (like monitoring_location_id for wells)
df_std['snotel_station_id'] = df_std['station_triplet']

# Reorder columns for consistency
keep_cols = [
    'snotel_station_id',
    'station_triplet',
    'station_id',
    'network_code',
    'station_name',
    'state_code',
    'county_name',
    'latitude',
    'longitude',
    'altitude',
    'time_zone',
    'hydrologic_unit_code',
    'huc8',
    'associated_hucs',
    'shef_id',
    'operator',
    'start_date',
    'end_date',
    'dco_code',
    'pedon_code',
    'active_flag'
]

df_final = df_std[[c for c in keep_cols if c in df_std.columns]].copy()

# Save standardized station metadata
std_file = f'{snotel_dir}/nrcs_snotel_stations_conus.csv'
df_final.to_csv(std_file, index=False)
print(f"\nSaved standardized station metadata: {std_file}")

# Save station manifest (for use in subsequent observation downloads)
manifest_file = f'{snotel_dir}/snotel_station_manifest.csv'
manifest_df = df_final[['station_triplet', 'active_flag', 'latitude', 'longitude', 'hydrologic_unit_code']].copy()
manifest_df.to_csv(manifest_file, index=False)
print(f"Saved station manifest: {manifest_file}")

# Summary statistics
print(f"\n=== SUMMARY ===")
print(f"Total unique SNOTEL stations: {len(df_final):,}")
print(f"Active stations: {df_final['active_flag'].sum():,}")
print(f"Inactive stations: {(~df_final['active_flag']).sum():,}")
print(f"\nState breakdown:")
print(df_final.groupby('state_code').size().sort_values(ascending=False).to_string())
print(f"\nHUC-8 availability: {df_final['huc8'].notna().sum():,} / {len(df_final):,} ({100*df_final['huc8'].notna().sum()/len(df_final):.1f}%)")
print(f"\nNetwork codes present: {df_final['network_code'].unique()}")

Status code: 200
Number of ACTIVE SNOTEL stations returned: 918

Status code: 200
Number of INACTIVE SNOTEL stations returned: 933

Duplicate stationTriplets found: 918
After removing duplicates: 933 unique stations

Total unique SNOTEL stations (active + inactive): 933
  - Active: 918
  - Inactive: 15

Saved raw station metadata: /content/drive/MyDrive/ecohydro_project/data/raw/snotel_metadata/nrcs_snotel_stations_raw.csv

Saved standardized station metadata: /content/drive/MyDrive/ecohydro_project/data/raw/snotel_metadata/nrcs_snotel_stations_conus.csv
Saved station manifest: /content/drive/MyDrive/ecohydro_project/data/raw/snotel_metadata/snotel_station_manifest.csv

=== SUMMARY ===
Total unique SNOTEL stations: 933
Active stations: 918
Inactive stations: 15

State breakdown:
state_code
UT    141
CO    120
MT     97
WY     90
AK     88
ID     87
OR     82
WA     79
NV     58
CA     36
NM     30
AZ     23
SD      2

HUC-8 availability: 933 / 933 (100.0%)

Network codes present: ['SNT

In [ ]:
# %% Step 2: Validate station metadata and inspect available elements
import pandas as pd
import numpy as np
import requests
import os

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'

# Load standardized station metadata
stations_file = f'{snotel_dir}/nrcs_snotel_stations_conus.csv'
stations = pd.read_csv(stations_file, dtype={'station_triplet': str})

print(f"Loaded {len(stations):,} SNOTEL stations")

# === VALIDATION CHECKS ===
print("\n=== VALIDATION CHECKS ===")

# 1. Check for duplicate station_triplet
n_dupes = stations['station_triplet'].duplicated().sum()
print(f"1. Duplicate station_triplet: {n_dupes} (should be 0)")

# 2. Check for missing coordinates
missing_coords = stations['latitude'].isna().sum() + stations['longitude'].isna().sum()
print(f"2. Stations with missing coordinates: {missing_coords}")

# 3. Check coordinate validity
valid_coords = (
    stations['latitude'].between(-90, 90) &
    stations['longitude'].between(-180, 180)
)
print(f"3. Stations with valid coordinates: {valid_coords.sum():,} / {len(stations):,}")

# 4. Check network code
unique_networks = stations['network_code'].unique()
print(f"4. Unique network codes: {unique_networks} (should be ['SNTL'])")

# 5. Active vs inactive breakdown
print(f"5. Active stations: {stations['active_flag'].sum():,}")
print(f"   Inactive stations: {(~stations['active_flag']).sum():,}")

# 6. Check for missing start_date or end_date
missing_dates = stations['start_date'].isna().sum() + stations['end_date'].isna().sum()
print(f"6. Stations with missing start_date or end_date: {missing_dates}")

# 7. Parse start_date and end_date to check for reasonable ranges
stations['start_date_parsed'] = pd.to_datetime(stations['start_date'], errors='coerce')
stations['end_date_parsed'] = pd.to_datetime(stations['end_date'], errors='coerce')

print(f"\n   Start date range: {stations['start_date_parsed'].min()} to {stations['start_date_parsed'].max()}")
print(f"   End date range: {stations['end_date_parsed'].min()} to {stations['end_date_parsed'].max()}")

# === INSPECT AVAILABLE ELEMENTS ===
print("\n=== INSPECT AVAILABLE ELEMENTS ===")

# Query stations with returnStationElements=true to see what elements are available
url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"

# Test with a small subset of stations first (e.g., first 10)
test_stations = stations['station_triplet'].head(10).tolist()
test_triplets = ','.join(test_stations)

print(f"Querying elements for {len(test_stations)} test stations...")

params = {
    "stationTriplets": test_triplets,
    "activeOnly": "false",
    "returnForecastPointMetadata": "false",
    "returnReservoirMetadata": "false",
    "returnStationElements": "true"  # This returns element information
}

try:
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data_with_elements = response.json()

    print(f"Status code: {response.status_code}")
    print(f"Number of stations returned: {len(data_with_elements)}")

    # Inspect elements for the first station
    if len(data_with_elements) > 0:
        first_station = data_with_elements[0]
        print(f"\nFirst station: {first_station.get('name', 'N/A')} ({first_station.get('stationTriplet', 'N/A')})")

        if 'stationElements' in first_station:
            elements = first_station['stationElements']
            print(f"Number of elements for this station: {len(elements)}")

            # Show element details
            print("\nAvailable elements:")
            element_df = pd.DataFrame(elements)
            if len(element_df) > 0:
                print(element_df[['elementCode', 'elementName', 'unit', 'duration']].to_string(index=False))

                # Save element inventory for all test stations
                all_elements = []
                for station in data_with_elements:
                    if 'stationElements' in station:
                        for elem in station['stationElements']:
                            elem['station_triplet'] = station['stationTriplet']
                            elem['station_name'] = station['name']
                            all_elements.append(elem)

                elements_inventory = pd.DataFrame(all_elements)
                if len(elements_inventory) > 0:
                    elements_file = f'{snotel_dir}/snotel_elements_inventory_test.csv'
                    elements_inventory.to_csv(elements_file, index=False)
                    print(f"\nSaved element inventory for test stations: {elements_file}")
                    print(f"Total station-element combinations in test: {len(elements_inventory):,}")

                    # Show most common elements
                    print("\nMost common elements across test stations:")
                    elem_counts = elements_inventory['elementName'].value_counts().head(20)
                    print(elem_counts.to_string())
        else:
            print("No 'stationElements' field found in response")
    else:
        print("No stations returned in test query")

except Exception as e:
    print(f"Error querying elements: {type(e).__name__}: {e}")
    print("This is OK for now - we'll handle element discovery during the full download")

# Save validated stations
validated_file = f'{snotel_dir}/nrcs_snotel_stations_validated.csv'
stations.to_csv(validated_file, index=False)
print(f"\nSaved validated station metadata: {validated_file}")

print("\n=== READY FOR NEXT STEP ===")
print("Station metadata validated. Next step will be to:")
print("1. Query the full element inventory for all stations")
print("2. Build a station-element manifest for observation downloads")

Loaded 933 SNOTEL stations

=== VALIDATION CHECKS ===
1. Duplicate station_triplet: 0 (should be 0)
2. Stations with missing coordinates: 0
3. Stations with valid coordinates: 933 / 933
4. Unique network codes: ['SNTL'] (should be ['SNTL'])
5. Active stations: 918
   Inactive stations: 15
6. Stations with missing start_date or end_date: 0

   Start date range: 1937-10-01 00:00:00 to 2026-08-29 14:00:00
   End date range: 1986-10-01 00:00:00 to 2100-01-01 00:00:00

=== INSPECT AVAILABLE ELEMENTS ===
Querying elements for 10 test stations...
Status code: 200
Number of stations returned: 10

First station: Adin Mtn (301:CA:SNTL)
Number of elements for this station: 73

Available elements:
Error querying elements: KeyError: "['elementName', 'unit', 'duration'] not in index"
This is OK for now - we'll handle element discovery during the full download

Saved validated station metadata: /content/drive/MyDrive/ecohydro_project/data/raw/snotel_metadata/nrcs_snotel_stations_validated.csv

=== RE

In [ ]:
# %% Step 2b: Inspect element structure and build full element inventory
import pandas as pd
import requests
import os

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'

# Load validated stations
stations_file = f'{snotel_dir}/nrcs_snotel_stations_validated.csv'
stations = pd.read_csv(stations_file, dtype={'station_triplet': str})

url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"

# Test with first 5 stations to see element structure
test_stations = stations['station_triplet'].head(5).tolist()
test_triplets = ','.join(test_stations)

print(f"Querying elements for {len(test_stations)} test stations...")

params = {
    "stationTriplets": test_triplets,
    "activeOnly": "false",
    "returnForecastPointMetadata": "false",
    "returnReservoirMetadata": "false",
    "returnStationElements": "true"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data_with_elements = response.json()

print(f"Status code: {response.status_code}")
print(f"Number of stations returned: {len(data_with_elements)}")

# Inspect the structure of stationElements
if len(data_with_elements) > 0 and 'stationElements' in data_with_elements[0]:
    first_station = data_with_elements[0]
    elements = first_station['stationElements']

    print(f"\nFirst station: {first_station.get('name', 'N/A')} ({first_station.get('stationTriplet', 'N/A')})")
    print(f"Number of elements: {len(elements)}")

    # Show the actual keys in the first element
    print("\nKeys in first element:")
    print(list(elements[0].keys()))

    # Convert to DataFrame and show all columns
    elem_df = pd.DataFrame(elements)
    print(f"\nElement DataFrame columns: {list(elem_df.columns)}")
    print(f"\nFirst 5 elements:")
    print(elem_df.head().to_string())

    # Save this test output
    test_file = f'{snotel_dir}/snotel_elements_structure_test.csv'
    elem_df.to_csv(test_file, index=False)
    print(f"\nSaved element structure test: {test_file}")
else:
    print("No stationElements found or structure unexpected")

Querying elements for 5 test stations...
Status code: 200
Number of stations returned: 5

First station: Adin Mtn (301:CA:SNTL)
Number of elements: 73

Keys in first element:
['elementCode', 'ordinal', 'durationName', 'dataPrecision', 'storedUnitCode', 'originalUnitCode', 'beginDate', 'endDate', 'derivedData']

Element DataFrame columns: ['elementCode', 'ordinal', 'durationName', 'dataPrecision', 'storedUnitCode', 'originalUnitCode', 'beginDate', 'endDate', 'derivedData']

First 5 elements:
  elementCode  ordinal   durationName  dataPrecision storedUnitCode originalUnitCode         beginDate           endDate  derivedData
0        BATT        1          DAILY              2           volt             volt  1984-09-14 06:00  2100-01-01 00:00        False
1        BATT        1         HOURLY              2           volt             volt  1984-09-14 06:00  2100-01-01 00:00        False
2        BATT        2          DAILY              2           volt             volt  2015-07-01 12:00

In [ ]:
# %% Step 2d: Build element inventory using elementCode and duration directly
import pandas as pd
import requests
import os

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'

# Load validated stations
stations_file = f'{snotel_dir}/nrcs_snotel_stations_validated.csv'
stations = pd.read_csv(stations_file, dtype={'station_triplet': str})

url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/stations"

# Query all 932 stations with returnStationElements=true
# We'll do this in batches to avoid timeout
print("Querying element inventory for ALL stations...")

all_elements = []
batch_size = 50

for batch_num, start in enumerate(range(0, len(stations), batch_size)):
    batch_stations = stations.iloc[start:start + batch_size]
    batch_triplets = ','.join(batch_stations['station_triplet'].tolist())

    params = {
        "stationTriplets": batch_triplets,
        "activeOnly": "false",
        "returnForecastPointMetadata": "false",
        "returnReservoirMetadata": "false",
        "returnStationElements": "true"
    }

    try:
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        data_batch = response.json()

        for station in data_batch:
            station_triplet = station['stationTriplet']
            station_name = station['name']

            if 'stationElements' in station:
                for elem in station['stationElements']:
                    element_record = {
                        'station_triplet': station_triplet,
                        'station_name': station_name,
                        'element_code': elem['elementCode'],
                        'duration': elem['durationName'],
                        'unit': elem.get('storedUnitCode', 'N/A'),
                        'begin_date': elem['beginDate'],
                        'end_date': elem['endDate']
                    }
                    all_elements.append(element_record)

        if batch_num % 5 == 0:
            print(f"Batch {batch_num}: processed {start + len(batch_stations):,}/{len(stations):,} stations")

    except Exception as e:
        print(f"Error in batch {batch_num}: {type(e).__name__}: {e}")
        # Continue to next batch

# Create inventory DataFrame
elements_inventory = pd.DataFrame(all_elements)

print(f"\n=== ELEMENT INVENTORY COMPLETE ===")
print(f"Total station-element-duration combinations: {len(elements_inventory):,}")

# Show sample
print("\nSample element inventory:")
print(elements_inventory[['station_triplet', 'element_code', 'duration', 'unit']].head(10).to_string(index=False))

# Show unique element codes
print("\nUnique element codes:")
unique_elements = elements_inventory['element_code'].value_counts().head(30)
print(unique_elements.to_string())

# Show unique durations
print("\nUnique durations:")
print(elements_inventory['duration'].unique())

# Save
elements_file = f'{snotel_dir}/snotel_elements_inventory_full.csv'
elements_inventory.to_csv(elements_file, index=False)
print(f"\nSaved element inventory: {elements_file}")

# Also save a summary by element code
element_summary = elements_inventory.groupby('element_code').agg(
    n_stations=('station_triplet', 'nunique'),
    n_combinations=('element_code', 'count')
).sort_values('n_stations', ascending=False)

summary_file = f'{snotel_dir}/snotel_element_summary.csv'
element_summary.to_csv(summary_file)
print(f"Saved element summary: {summary_file}")
print("\nElement summary (top 20):")
print(element_summary.head(20).to_string())

Querying element inventory for ALL stations...
Batch 0: processed 50/933 stations
Batch 5: processed 300/933 stations
Batch 10: processed 550/933 stations
Batch 15: processed 800/933 stations

=== ELEMENT INVENTORY COMPLETE ===
Total station-element-duration combinations: 139,555

Sample element inventory:
station_triplet element_code      duration unit
    301:CA:SNTL         BATT         DAILY volt
    301:CA:SNTL         BATT        HOURLY volt
    301:CA:SNTL         BATT         DAILY volt
    301:CA:SNTL         BATT        HOURLY volt
    301:CA:SNTL         PRCP CALENDAR_YEAR   in
    301:CA:SNTL         PRCP         DAILY   in
    301:CA:SNTL         PRCP   SEMIMONTHLY   in
    301:CA:SNTL         PRCP       MONTHLY   in
    301:CA:SNTL         PRCP    WATER_YEAR   in
    301:CA:SNTL      PRCPMTD         DAILY   in

Unique element codes:
element_code
TAVG       9382
TMAX       9375
TMIN       9370
TOBS       7533
STO        7248
SMS        7048
PREC       5990
ZDUM       5944


In [ ]:
# %% Quick count: How many DAILY observations to download?
import pandas as pd

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'

elements_file = f'{snotel_dir}/snotel_elements_inventory_full.csv'
elements = pd.read_csv(elements_file)

# Filter to DAILY duration only
daily_elements = elements[elements['duration'] == 'DAILY'].copy()

print(f"Total station-element combinations: {len(elements):,}")
print(f"DAILY duration combinations: {len(daily_elements):,}")
print(f"Unique elements with DAILY data: {daily_elements['element_code'].nunique()}")
print(f"Unique stations with DAILY data: {daily_elements['station_triplet'].nunique()}")

print("\nTop 30 elements by station count (DAILY only):")
elem_counts = daily_elements.groupby('element_code')['station_triplet'].nunique().sort_values(ascending=False).head(30)
print(elem_counts.to_string())

Total station-element combinations: 139,555
DAILY duration combinations: 40,922
Unique elements with DAILY data: 46
Unique stations with DAILY data: 932

Top 30 elements by station count (DAILY only):
element_code
TAVG       932
TMIN       932
TMAX       932
PREC       930
TOBS       926
PRCPMTD    925
PRCP       924
BATT       923
SNWD       923
WTEQ       898
SNRR       896
SNDN       893
PRCPSA     892
STN        520
STV        520
STO        520
STX        520
SMX        518
SMN        518
SMV        518
SMS        518
DIAG       422
SAL        359
RDC        359
ZDUM       307
BATX       203
RHUM        93
WSPDV       69
WSPDX       60
WDIRV       60


In [ ]:
# %% Step 3: Download all daily SNOTEL observations with checkpointing
import requests
import pandas as pd
import os
import time
from datetime import datetime

base = '/content/drive/MyDrive/ecohydro_project'
snotel_dir = f'{base}/data/raw/snotel_metadata'
observations_dir = f'{base}/data/raw/snotel_observations/daily'
os.makedirs(observations_dir, exist_ok=True)

# Load element inventory and filter to DAILY only
elements_file = f'{snotel_dir}/snotel_elements_inventory_full.csv'
elements = pd.read_csv(elements_file)
daily_elements = elements[elements['duration'] == 'DAILY'].copy()

# Create unique station-element combinations (some stations have duplicate element entries)
daily_unique = daily_elements.drop_duplicates(subset=['station_triplet', 'element_code']).copy()
daily_unique = daily_unique.sort_values(['station_triplet', 'element_code']).reset_index(drop=True)

print(f"Total unique station-element combinations to download: {len(daily_unique):,}")

# AWDB API endpoint for station data (includes observations)
# We use the /data endpoint to retrieve actual observations
data_url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/data"

# Checkpoint log
checkpoint_file = f'{observations_dir}/download_checkpoint_log.csv'

def load_checkpoint():
    """Load checkpoint log if it exists."""
    if os.path.exists(checkpoint_file):
        return pd.read_csv(checkpoint_file, dtype={'station_triplet': str, 'element_code': str})
    return pd.DataFrame(columns=['batch_num', 'station_triplet', 'element_code', 'status', 'n_observations', 'timestamp'])

def save_checkpoint(batch_records):
    """Append batch records to checkpoint log."""
    df = pd.DataFrame(batch_records)
    df.to_csv(checkpoint_file, index=False, mode='a', header=not os.path.exists(checkpoint_file))

# Load checkpoint to see what's already downloaded
checkpoint_df = load_checkpoint()
completed_combinations = set()
if len(checkpoint_df) > 0:
    completed_mask = checkpoint_df['status'] == 'completed'
    completed = checkpoint_df[completed_mask][['station_triplet', 'element_code']].drop_duplicates()
    completed_combinations = set(zip(completed['station_triplet'], completed['element_code']))
    print(f"Already completed: {len(completed_combinations):,} station-element combinations")

# Filter to only combinations not yet completed
remaining = daily_unique[
    ~daily_unique.apply(lambda row: (row['station_triplet'], row['element_code']) in completed_combinations, axis=1)
].reset_index(drop=True)

print(f"Remaining to download: {len(remaining):,} station-element combinations")

if len(remaining) == 0:
    print("All daily observations already downloaded!")
else:
    print(f"\nStarting download of {len(remaining):,} station-element combinations...")

    # Download in batches
    start_time = time.time()
    batch_size = 100  # Download 100 station-elements per batch

    for batch_num, start in enumerate(range(0, len(remaining), batch_size)):
        batch = remaining.iloc[start:start + batch_size]
        batch_records = []

        for idx, row in batch.iterrows():
            station_triplet = row['station_triplet']
            element_code = row['element_code']

            try:
                # Query daily data for this station-element
                params = {
                    "stationTriplets": station_triplet,
                    "elements": element_code,
                    "beginDate": "1900-01-01",
                    "endDate": "2100-12-31",
                    "getFlags": "true"
                }

                response = requests.get(data_url, params=params, timeout=60)
                response.raise_for_status()
                data = response.json()

                # Parse observations
                n_obs = 0
                if len(data) > 0 and 'data' in data[0]:
                    observations = data[0]['data']
                    n_obs = len(observations)

                    if n_obs > 0:
                        # Convert to DataFrame and save
                        obs_df = pd.DataFrame(observations)
                        obs_df['station_triplet'] = station_triplet
                        obs_df['element_code'] = element_code

                        # Save to individual file
                        safe_station = station_triplet.replace(':', '_')
                        obs_file = f'{observations_dir}/obs_{safe_station}_{element_code}.csv'
                        obs_df.to_csv(obs_file, index=False)

                batch_records.append({
                    'batch_num': batch_num,
                    'station_triplet': station_triplet,
                    'element_code': element_code,
                    'status': 'completed',
                    'n_observations': n_obs,
                    'timestamp': datetime.now().isoformat()
                })

            except Exception as e:
                batch_records.append({
                    'batch_num': batch_num,
                    'station_triplet': station_triplet,
                    'element_code': element_code,
                    'status': f'error: {type(e).__name__}',
                    'n_observations': 0,
                    'timestamp': datetime.now().isoformat()
                })

        # Save checkpoint after each batch
        save_checkpoint(batch_records)

        # Progress report
        if batch_num % 10 == 0:
            elapsed = time.time() - start_time
            n_done = start + len(batch)
            rate = n_done / elapsed if elapsed > 0 else 0
            eta_min = (len(remaining) - n_done) / rate / 60 if rate > 0 else 0
            print(f"[Batch {batch_num}] {n_done:,}/{len(remaining):,} station-elements "
                  f"({rate:.1f}/sec, ETA: {eta_min:.0f} min)")

    print(f"\nDownload complete for {len(remaining):,} station-element combinations")
    total_time = time.time() - start_time
    print(f"Total runtime: {total_time/60:.1f} minutes")

Total unique station-element combinations to download: 18,234
Already completed: 18,226 station-element combinations
Remaining to download: 8 station-element combinations

Starting download of 8 station-element combinations...
[Batch 0] 8/8 station-elements (0.3/sec, ETA: 0 min)

Download complete for 8 station-element combinations
Total runtime: 0.5 minutes


In [ ]:
# %% Step 4 (memory-efficient): Combine observations in batches with incremental writes
import pandas as pd
import glob
import os
import json

base = '/content/drive/MyDrive/ecohydro_project'
observations_dir = f'{base}/data/raw/snotel_observations/daily'
output_dir = f'{observations_dir}/combined_batches'
os.makedirs(output_dir, exist_ok=True)

# Find all observation files
obs_files = glob.glob(f'{observations_dir}/obs_*.csv')
print(f"Found {len(obs_files):,} observation files")

# Process in batches of 500 files, save each batch
batch_size = 500
all_obs_count = 0

for batch_num, start in enumerate(range(0, len(obs_files), batch_size)):
    batch_files = obs_files[start:start + batch_size]
    batch_obs = []

    for obs_file in batch_files:
        try:
            df = pd.read_csv(obs_file)

            # Parse values column more efficiently
            for idx, row in df.iterrows():
                station_triplet = row['station_triplet']
                element_code = row['element_code']

                try:
                    # Parse the values list
                    values_str = row['values'].replace("'", '"')  # Convert to valid JSON
                    values_list = json.loads(values_str)

                    for obs in values_list:
                        batch_obs.append({
                            'station_triplet': station_triplet,
                            'element_code': element_code,
                            'date': obs.get('date'),
                            'value': obs.get('value')
                        })
                except:
                    pass  # Skip malformed rows

        except Exception as e:
            pass  # Skip problematic files

    # Save this batch
    if len(batch_obs) > 0:
        batch_df = pd.DataFrame(batch_obs)
        batch_file = f'{output_dir}/batch_{batch_num:03d}.csv'
        batch_df.to_csv(batch_file, index=False)
        all_obs_count += len(batch_obs)
        print(f"Batch {batch_num}: saved {len(batch_obs):,} observations ({all_obs_count:,} total)")

    # Clear memory
    del batch_obs
    del batch_df

print(f"\n=== ALL BATCHES COMPLETE ===")
print(f"Total observations across all batches: {all_obs_count:,}")
print(f"Batch files saved to: {output_dir}")

# Now combine all batches
print("\n=== COMBINING ALL BATCHES ===")
batch_files = glob.glob(f'{output_dir}/batch_*.csv')

combined_dfs = []
for i, batch_file in enumerate(batch_files):
    df = pd.read_csv(batch_file)
    combined_dfs.append(df)
    if i % 5 == 0:
        print(f"Loaded {i+1}/{len(batch_files)} batches...")

master_obs = pd.concat(combined_dfs, ignore_index=True)

# Save master file
master_file = f'{observations_dir}/snotel_daily_observations_master.csv'
master_obs.to_csv(master_file, index=False)

print(f"\n=== MASTER DATASET COMPLETE ===")
print(f"Total observation rows: {len(master_obs):,}")
print(f"Unique stations: {master_obs['station_triplet'].nunique():,}")
print(f"Unique elements: {master_obs['element_code'].nunique()}")
print(f"Date range: {master_obs['date'].min()} to {master_obs['date'].max()}")
print(f"Saved: {master_file}")

Found 11,209 observation files
Batch 0: saved 4,774,723 observations (4,774,723 total)
Batch 1: saved 4,106,390 observations (8,881,113 total)
Batch 2: saved 6,457,119 observations (15,338,232 total)
Batch 3: saved 5,776,420 observations (21,114,652 total)
Batch 4: saved 6,480,370 observations (27,595,022 total)
Batch 5: saved 6,170,646 observations (33,765,668 total)
Batch 6: saved 6,356,257 observations (40,121,925 total)
Batch 7: saved 6,431,912 observations (46,553,837 total)
Batch 8: saved 6,345,969 observations (52,899,806 total)
Batch 9: saved 6,190,817 observations (59,090,623 total)
Batch 10: saved 6,555,156 observations (65,645,779 total)
Batch 11: saved 6,334,600 observations (71,980,379 total)
Batch 12: saved 6,488,649 observations (78,469,028 total)
Batch 13: saved 6,352,216 observations (84,821,244 total)
Batch 14: saved 5,810,144 observations (90,631,388 total)
Batch 15: saved 6,528,787 observations (97,160,175 total)
Batch 16: saved 1,903,678 observations (99,063,853 to

In [ ]:
# %% SNOTEL QA: inspect future dates without loading the master CSV into RAM

import os
import pandas as pd

base = "/content/drive/MyDrive/ecohydro_project"
observations_dir = f"{base}/data/raw/snotel_observations/daily"
master_file = f"{observations_dir}/snotel_daily_observations_master.csv"

if not os.path.isfile(master_file):
    raise FileNotFoundError(master_file)

cutoff = pd.Timestamp("2026-09-26")  # project QA cutoff; change if needed
chunk_size = 100_000
sample_limit = 20

total_rows = 0
future_rows = 0
invalid_date_rows = 0
first_date = None
last_date = None
future_sample = []
future_by_element = {}
future_by_station = {}

reader = pd.read_csv(
    master_file,
    usecols=["station_triplet", "element_code", "date", "value"],
    dtype={
        "station_triplet": "string",
        "element_code": "string",
        "date": "string",
        "value": "string",
    },
    chunksize=chunk_size,
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)

    dates = pd.to_datetime(chunk["date"], format="%Y-%m-%d", errors="coerce")
    invalid_date_rows += int(dates.isna().sum())

    valid_dates = dates.dropna()
    if not valid_dates.empty:
        chunk_first = valid_dates.min()
        chunk_last = valid_dates.max()
        first_date = chunk_first if first_date is None else min(first_date, chunk_first)
        last_date = chunk_last if last_date is None else max(last_date, chunk_last)

    future_mask = dates.gt(cutoff).fillna(False)
    future = chunk.loc[future_mask]
    future_rows += len(future)

    if len(future_sample) < sample_limit:
        future_sample.extend(
            future.head(sample_limit - len(future_sample))
                  .to_dict("records")
        )

    for element, count in future["element_code"].value_counts().items():
        future_by_element[element] = future_by_element.get(element, 0) + int(count)

    for station, count in future["station_triplet"].value_counts().items():
        future_by_station[station] = future_by_station.get(station, 0) + int(count)

    if chunk_number % 100 == 0:
        print(f"Checked {total_rows:,} rows...")

print(f"\nTotal rows checked: {total_rows:,}")
print(f"Invalid or unparseable dates: {invalid_date_rows:,}")
print(f"Rows after {cutoff.date()}: {future_rows:,}")
print(f"Observed date range: {first_date} to {last_date}")

print("\nSample of future-dated rows:")
print(pd.DataFrame(future_sample).to_string(index=False))

print("\nFuture rows by element, top 10:")
print(pd.Series(future_by_element).sort_values(ascending=False).head(10).to_string())

print("\nFuture rows by station, top 10:")
print(pd.Series(future_by_station).sort_values(ascending=False).head(10).to_string())

Checked 10,000,000 rows...
Checked 20,000,000 rows...
Checked 30,000,000 rows...
Checked 40,000,000 rows...
Checked 50,000,000 rows...
Checked 60,000,000 rows...
Checked 70,000,000 rows...
Checked 80,000,000 rows...
Checked 90,000,000 rows...
Checked 100,000,000 rows...
Checked 110,000,000 rows...

Total rows checked: 112,954,865
Invalid or unparseable dates: 0
Rows after 2026-09-26: 5
Observed date range: 1963-09-30 00:00:00 to 2038-08-04 00:00:00

Sample of future-dated rows:
station_triplet element_code       date value
    518:CA:SNTL         WTEQ 2038-08-04   0.0
   1120:CO:SNTL         TAVG 2028-12-01  None
   1120:CO:SNTL         TMAX 2028-12-01  None
   1120:CO:SNTL         TMIN 2028-12-01  None
   1120:CO:SNTL         WTEQ 2028-12-01   0.0

Future rows by element, top 10:
WTEQ    2
TAVG    1
TMAX    1
TMIN    1

Future rows by station, top 10:
1120:CO:SNTL    4
518:CA:SNTL     1


In [ ]:
# %% SNOTEL checkpoint audit — safe to run after a Colab restart
from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/ecohydro_project")
daily_dir = base / "data/raw/snotel_observations/daily"
checkpoint_path = daily_dir / "download_checkpoint_log.csv"

log = pd.read_csv(
    checkpoint_path,
    dtype={"station_triplet": "string", "element_code": "string", "status": "string"}
)

print("Checkpoint rows:", len(log))
print("\nStatus counts:")
print(log["status"].value_counts(dropna=False).to_string())

print("\nCompleted rows by recorded observation count:")
completed = log.loc[log["status"].eq("completed")].copy()
completed["n_observations"] = pd.to_numeric(
    completed["n_observations"], errors="coerce"
)
print("Completed with n_observations = 0:",
      int(completed["n_observations"].eq(0).sum()))
print("Completed with n_observations > 0:",
      int(completed["n_observations"].gt(0).sum()))
print("Completed with missing/unparseable count:",
      int(completed["n_observations"].isna().sum()))

# Use the latest checkpoint entry for each station–element pair.
latest = log.drop_duplicates(
    ["station_triplet", "element_code"], keep="last"
).copy()

def expected_file(row):
    safe_station = str(row["station_triplet"]).replace(":", "_")
    return daily_dir / f"obs_{safe_station}_{row['element_code']}.csv"

latest["file_exists"] = latest.apply(
    lambda row: expected_file(row).is_file(), axis=1
)

print("\nLatest status versus saved file:")
print(pd.crosstab(
    latest["status"], latest["file_exists"], dropna=False
).to_string())

print("\nCompleted with reported observations but no file:")
suspect = latest[
    latest["status"].eq("completed")
    & pd.to_numeric(latest["n_observations"], errors="coerce").gt(0)
    & ~latest["file_exists"]
]
print(suspect[
    ["station_triplet", "element_code", "n_observations"]
].head(20).to_string(index=False))
print("Total:", len(suspect))

Checkpoint rows: 18260

Status counts:
status
completed             18233
error: HTTPError         26
error: ReadTimeout        1

Completed rows by recorded observation count:
Completed with n_observations = 0: 7024
Completed with n_observations > 0: 11209
Completed with missing/unparseable count: 0

Latest status versus saved file:
file_exists       False  True 
status                        
completed          7024  11209
error: HTTPError      1      0

Completed with reported observations but no file:
Empty DataFrame
Columns: [station_triplet, element_code, n_observations]
Index: []
Total: 0


In [ ]:
# %% Diagnostic only: inspect three "completed, zero observations" requests
import pandas as pd
import requests
from pathlib import Path

daily_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/snotel_observations/daily"
)
log = pd.read_csv(
    daily_dir / "download_checkpoint_log.csv",
    dtype={"station_triplet": str, "element_code": str, "status": str}
)

latest = log.drop_duplicates(
    ["station_triplet", "element_code"], keep="last"
).copy()
latest["n_observations"] = pd.to_numeric(
    latest["n_observations"], errors="coerce"
)

zero_sample = latest[
    latest["status"].eq("completed")
    & latest["n_observations"].eq(0)
].sample(n=3, random_state=42)

url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/data"

for row in zero_sample.itertuples(index=False):
    params = {
        "stationTriplets": row.station_triplet,
        "elements": row.element_code,
        "duration": "DAILY",
        "beginDate": "2000-01-01",
        "endDate": "2000-01-10",
        "getFlags": "true",
    }

    try:
        response = requests.get(url, params=params, timeout=60)
        print(f"\n{row.station_triplet} | {row.element_code}")
        print("HTTP status:", response.status_code)
        response.raise_for_status()

        payload = response.json()
        print("Top-level type:", type(payload).__name__)
        print("Number of top-level items:",
              len(payload) if isinstance(payload, (list, dict)) else "N/A")

        if isinstance(payload, list) and payload:
            first = payload[0]
            print("First-item keys:",
                  list(first) if isinstance(first, dict) else type(first).__name__)
            if isinstance(first, dict):
                print("First-item structure:",
                      {key: type(value).__name__ for key, value in first.items()})
                print("Values length:",
                      len(first.get("values") or [])
                      if isinstance(first.get("values"), list) else "N/A")
        else:
            print("Response preview:", str(payload)[:500])

    except requests.RequestException as exc:
        print("Request failed:", repr(exc))


924:MT:SNTL | ZDUM
HTTP status: 200
Top-level type: list
Number of top-level items: 0
Response preview: []

312:ID:SNTL | STX
HTTP status: 200
Top-level type: list
Number of top-level items: 0
Response preview: []

373:NV:SNTL | SAL
HTTP status: 200
Top-level type: list
Number of top-level items: 0
Response preview: []


In [ ]:
# %% Verify original download parser using one saved and one zero-result combination
import pandas as pd
import requests
from pathlib import Path

daily_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/snotel_observations/daily"
)
log = pd.read_csv(
    daily_dir / "download_checkpoint_log.csv",
    dtype={"station_triplet": str, "element_code": str, "status": str}
)
latest = log.drop_duplicates(
    ["station_triplet", "element_code"], keep="last"
).copy()
latest["n_observations"] = pd.to_numeric(
    latest["n_observations"], errors="coerce"
)

known = latest[
    latest["status"].eq("completed")
    & latest["n_observations"].gt(0)
].iloc[0]

zero = latest[
    latest["status"].eq("completed")
    & latest["n_observations"].eq(0)
].iloc[0]

url = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/data"

for label, row in [("known saved", known), ("reported zero", zero)]:
    params = {
        "stationTriplets": row["station_triplet"],
        "elements": row["element_code"],
        "beginDate": "1900-01-01",
        "endDate": "2100-12-31",
        "getFlags": "true",
    }

    print(f"\n{label}: {row['station_triplet']} | {row['element_code']}")
    try:
        response = requests.get(url, params=params, timeout=120)
        print("HTTP status:", response.status_code)
        response.raise_for_status()
        payload = response.json()

        print("Top-level type:", type(payload).__name__)
        print("Top-level item count:", len(payload) if isinstance(payload, list) else "N/A")

        if isinstance(payload, list):
            for i, item in enumerate(payload[:5]):
                print(
                    f"Item {i}: keys={list(item) if isinstance(item, dict) else 'not a dict'}, "
                    f"len(data)={len(item.get('data') or []) if isinstance(item, dict) else 'N/A'}, "
                    f"len(values)={len(item.get('values') or []) if isinstance(item, dict) else 'N/A'}"
                )
    except requests.RequestException as exc:
        print("Request failed:", repr(exc))


known saved: 1000:OR:SNTL | PRCP
HTTP status: 200
Top-level type: list
Top-level item count: 1
Item 0: keys=['stationTriplet', 'data'], len(data)=1, len(values)=0

reported zero: 1000:OR:SNTL | BATT
HTTP status: 200
Top-level type: list
Top-level item count: 0


In [ ]:
# %% Inspect one saved SNOTEL file and its checkpoint entry
from pathlib import Path
import pandas as pd
import ast

daily_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/snotel_observations/daily"
)
station = "1000:OR:SNTL"
element = "PRCP"
file_path = daily_dir / f"obs_{station.replace(':', '_')}_{element}.csv"

log = pd.read_csv(
    daily_dir / "download_checkpoint_log.csv",
    dtype={"station_triplet": str, "element_code": str}
)
matching_log = log[
    log["station_triplet"].eq(station)
    & log["element_code"].eq(element)
]

print("Checkpoint entries:")
print(matching_log.to_string(index=False))
print("\nSaved file exists:", file_path.is_file())

series = pd.read_csv(file_path)
print("Series rows in file:", len(series))
print("Columns:", list(series.columns))

for i, row in series.head(5).iterrows():
    station_element = ast.literal_eval(row["stationElement"])
    values = ast.literal_eval(row["values"])
    print(
        f"Series {i}: element={station_element.get('elementCode')}, "
        f"ordinal={station_element.get('ordinal')}, "
        f"duration={station_element.get('durationName')}, "
        f"values={len(values):,}, "
        f"first={values[0] if values else None}, "
        f"last={values[-1] if values else None}"
    )

Checkpoint entries:
 batch_num station_triplet element_code    status  n_observations                  timestamp
         0    1000:OR:SNTL         PRCP completed               1 2026-09-22T16:58:19.492543

Saved file exists: True
Series rows in file: 1
Columns: ['stationElement', 'values', 'station_triplet', 'element_code']
Series 0: element=PRCP, ordinal=1, duration=DAILY, values=9,479, first={'date': '2000-10-01', 'value': 0.1}, last={'date': '2026-09-21', 'value': 0.1}


In [ ]:
# %% Audit saved SNOTEL series files without loading all observations
from pathlib import Path
import ast
import csv
import json
import pandas as pd

base = Path("/content/drive/MyDrive/ecohydro_project")
daily_dir = base / "data/raw/snotel_observations/daily"
audit_path = daily_dir / "snotel_saved_files_audit.csv"

obs_files = sorted(daily_dir.glob("obs_*.csv"))
print(f"Files to audit: {len(obs_files):,}")

def parse_nested(value):
    if pd.isna(value):
        return None
    text = str(value)
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return json.loads(text)

fieldnames = [
    "filename", "station_triplet", "element_code", "series_rows",
    "parsed_series", "daily_series", "non_daily_series",
    "total_values", "dated_values", "first_date", "last_date",
    "parse_errors", "error_example"
]

with open(audit_path, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fieldnames)
    writer.writeheader()

    for file_number, path in enumerate(obs_files, start=1):
        result = {
            "filename": path.name,
            "station_triplet": "",
            "element_code": "",
            "series_rows": 0,
            "parsed_series": 0,
            "daily_series": 0,
            "non_daily_series": 0,
            "total_values": 0,
            "dated_values": 0,
            "first_date": "",
            "last_date": "",
            "parse_errors": 0,
            "error_example": "",
        }

        try:
            # One source file at a time; its rows are series, not daily values.
            source = pd.read_csv(path, dtype=str)
            result["series_rows"] = len(source)

            dates_min = None
            dates_max = None

            for row in source.itertuples(index=False):
                try:
                    result["station_triplet"] = row.station_triplet
                    result["element_code"] = row.element_code

                    meta = parse_nested(row.stationElement)
                    values = parse_nested(row.values)

                    if not isinstance(meta, dict):
                        raise ValueError("stationElement is not a dictionary")
                    if not isinstance(values, list):
                        raise ValueError("values is not a list")

                    result["parsed_series"] += 1
                    if meta.get("durationName") == "DAILY":
                        result["daily_series"] += 1
                    else:
                        result["non_daily_series"] += 1

                    result["total_values"] += len(values)

                    for observation in values:
                        if not isinstance(observation, dict):
                            continue
                        date = observation.get("date")
                        if date:
                            result["dated_values"] += 1
                            # AWDB dates here have ISO YYYY-MM-DD form.
                            if dates_min is None or date < dates_min:
                                dates_min = date
                            if dates_max is None or date > dates_max:
                                dates_max = date

                except Exception as exc:
                    result["parse_errors"] += 1
                    if not result["error_example"]:
                        result["error_example"] = (
                            f"{type(exc).__name__}: {str(exc)[:180]}"
                        )

            result["first_date"] = dates_min or ""
            result["last_date"] = dates_max or ""

        except Exception as exc:
            result["parse_errors"] += 1
            result["error_example"] = (
                f"FILE ERROR {type(exc).__name__}: {str(exc)[:180]}"
            )

        writer.writerow(result)

        if file_number % 1000 == 0:
            print(f"Audited {file_number:,}/{len(obs_files):,} files")

print(f"\nSaved compact audit: {audit_path}")

# This audit file is small enough to summarize in memory.
audit = pd.read_csv(audit_path)
print("\nAudit totals:")
print(audit[
    ["series_rows", "parsed_series", "daily_series",
     "non_daily_series", "total_values", "dated_values", "parse_errors"]
].sum().to_string())
print("Files with errors:", int(audit["parse_errors"].gt(0).sum()))
print("Files containing non-daily series:",
      int(audit["non_daily_series"].gt(0).sum()))
print("Date range:", audit["first_date"].min(), "to", audit["last_date"].max())

print("\nExample problem files:")
print(audit.loc[
    audit["parse_errors"].gt(0) | audit["non_daily_series"].gt(0),
    ["filename", "parse_errors", "non_daily_series", "error_example"]
].head(10).to_string(index=False))

Files to audit: 11,209
Audited 1,000/11,209 files
Audited 2,000/11,209 files
Audited 3,000/11,209 files
Audited 4,000/11,209 files
Audited 5,000/11,209 files
Audited 6,000/11,209 files
Audited 7,000/11,209 files
Audited 8,000/11,209 files
Audited 9,000/11,209 files
Audited 10,000/11,209 files
Audited 11,000/11,209 files

Saved compact audit: /content/drive/MyDrive/ecohydro_project/data/raw/snotel_observations/daily/snotel_saved_files_audit.csv

Audit totals:
series_rows             11209
parsed_series           11209
daily_series            11209
non_daily_series            0
total_values        112954865
dated_values        112954865
parse_errors                0
Files with errors: 0
Files containing non-daily series: 0
Date range: 1963-09-30 to 2038-08-04

Example problem files:
Empty DataFrame
Columns: [filename, parse_errors, non_daily_series, error_example]
Index: []


###GRIDMET-----------------------------------------------

In [ ]:
import ee

ee.Initialize(project="ecohydro-project")

gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")

test = (
    gridmet
    .filterDate("2020-01-01", "2020-02-01")
    .select("pr")
)

print("January 2020 daily images:", test.size().getInfo())
print("First image bands:", ee.Image(test.first()).bandNames().getInfo())
print("First image status:", ee.Image(test.first()).get("status").getInfo())

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


January 2020 daily images: 31
First image bands: ['pr']
First image status: permanent


In [ ]:
import ee
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ee.Initialize(project="ecohydro-project")

PROJECT_DIR = Path("/content/drive/MyDrive/ecohydro_project/data/raw/gridmet")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Earth Engine's folder argument is a Drive folder name, not a path.
DRIVE_FOLDER = "ecohydro_gridmet_export"
EXPORT_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER

HUC2_CODES = [
    "01", "02", "03", "04", "05", "07", "08", "09", "10",
    "11", "12", "13", "14", "15", "16", "17", "18"
]

basins = (
    ee.FeatureCollection("USGS/WBD/2017/HUC02")
    .filter(ee.Filter.inList("huc2", HUC2_CODES))
    .select(["huc2", "name"])
)

gridmet = (
    ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
    .select(["pr", "eto", "etr"])
)

native_projection = ee.Image(
    gridmet.filterDate("2020-02-01", "2020-03-01").first()
).select("pr").projection()

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(),
    sharedInputs=True
)

print("HUC02 basins:", basins.size().getInfo())
print("Native grid:", native_projection.getInfo())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HUC02 basins: 17
Native grid: {'type': 'Projection', 'wkt': 'GEOGCS["unknown", \n  DATUM["unknown", \n    SPHEROID["Spheroid", 6378137.0, 298.257223563]], \n  PRIMEM["Greenwich", 0.0], \n  UNIT["degree", 0.017453292519943295], \n  AXIS["Longitude", EAST], \n  AXIS["Latitude", NORTH]]', 'transform': [0.041666666666666664, 0, -124.78749996666667, 0, -0.041666666666666664, 49.42083333333334]}


In [ ]:
def export_filename(year):
    if year == 2025:
        return "gridmet_monthly_17huc02_2025_01_09_fixedgrid.csv"
    return f"gridmet_monthly_17huc02_{year}_fixedgrid.csv"


def build_month_rows(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    daily = gridmet.filterDate(start, end)

    monthly_image = daily.sum().setDefaultProjection(native_projection)

    reduced = monthly_image.reduceRegions(
        collection=basins,
        reducer=reducer,
        crs=native_projection,
        tileScale=4
    )

    image_count = daily.size()
    statuses = ee.List(
        daily.aggregate_array("status")
    ).distinct().join(",")

    def clean_row(feature):
        feature = ee.Feature(feature)

        return ee.Feature(None, {
            "huc2": feature.get("huc2"),
            "name": feature.get("name"),
            "year": year,
            "month": month,
            "n_daily_images": image_count,
            "pr_mm_month": feature.get("pr_mean"),
            "eto_mm_month": feature.get("eto_mean"),
            "etr_mm_month": feature.get("etr_mean"),
            "pr_pixel_count": feature.get("pr_count"),
            "eto_pixel_count": feature.get("eto_count"),
            "etr_pixel_count": feature.get("etr_count"),
            "status_values": statuses,
            "product": "IDAHO_EPSCOR/GRIDMET",
            "grid": "native_GRIDMET_projection"
        })

    return reduced.map(clean_row)


def build_year_rows(year):
    last_month = 9 if year == 2025 else 12
    parts = [
        build_month_rows(year, month)
        for month in range(1, last_month + 1)
    ]

    combined = parts[0]
    for part in parts[1:]:
        combined = combined.merge(part)

    return combined


def submit_year(year):
    if not 1998 <= year <= 2025:
        raise ValueError("Year must be between 1998 and 2025.")

    filename = export_filename(year)
    project_file = PROJECT_DIR / filename
    export_file = EXPORT_DIR / filename

    if project_file.exists() and project_file.stat().st_size > 0:
        print(year, "already in project folder; skipped")
        return None

    if export_file.exists() and export_file.stat().st_size > 0:
        print(year, "already exported to Drive; skipped")
        return None

    prefix = filename[:-4]

    task = ee.batch.Export.table.toDrive(
        collection=build_year_rows(year),
        description=prefix,
        folder=DRIVE_FOLDER,
        fileNamePrefix=prefix,
        fileFormat="CSV",
        selectors=[
            "huc2", "name", "year", "month",
            "n_daily_images",
            "pr_mm_month", "eto_mm_month", "etr_mm_month",
            "pr_pixel_count", "eto_pixel_count", "etr_pixel_count",
            "status_values", "product", "grid"
        ]
    )

    task.start()
    print(year, "submitted:", task.id, task.status()["state"])
    return task

In [ ]:
test_2020_path = EXPORT_DIR / export_filename(2020)

if not test_2020_path.is_file():
    raise FileNotFoundError(
        f"2020 export not visible yet: {test_2020_path}"
    )

test_2020 = pd.read_csv(
    test_2020_path,
    dtype={"huc2": "string"}
)

print("Rows (expect 204):", len(test_2020))
print("Basins (expect 17):", test_2020["huc2"].nunique())
print("Months (expect 12):", sorted(test_2020["month"].unique()))
print("Duplicates:", test_2020.duplicated(["huc2", "year", "month"]).sum())
print("Missing precipitation:", test_2020["pr_mm_month"].isna().sum())
print("Statuses:", test_2020["status_values"].value_counts(dropna=False).to_dict())

feb_2020 = pd.read_csv(
    PROJECT_DIR / "gridmet_monthly_17huc02_2020_02_fixedgrid.csv",
    dtype={"huc2": "string"}
)

year_feb = test_2020.loc[
    test_2020["month"] == 2,
    ["huc2", "pr_mm_month", "eto_mm_month", "etr_mm_month"]
].set_index("huc2")

test_feb = feb_2020[
    ["huc2", "pr_mm_month", "eto_mm_month", "etr_mm_month"]
].set_index("huc2")

difference = year_feb - test_feb
print("Max February difference by band (mm):")
print(difference.abs().max().to_dict())

Rows (expect 204): 204
Basins (expect 17): 17
Months (expect 12): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Duplicates: 0
Missing precipitation: 0
Statuses: {'permanent': 204}
Max February difference by band (mm):
{'pr_mm_month': 3.552713678800501e-15, 'eto_mm_month': 7.105427357601002e-15, 'etr_mm_month': 1.4210854715202004e-14}


In [ ]:
tasks_by_year = {}

for year in range(1998, 2026):
    if year == 2020:
        continue

    task = submit_year(year)
    if task is not None:
        tasks_by_year[year] = task

print("New tasks submitted:", len(tasks_by_year))
print("Years submitted:", sorted(tasks_by_year))

1998 submitted: Z7RA7CIEGKXWPBJUAEFZSSOX READY
1999 submitted: 4YHDUFRQ5J567TAT6HEPDQCH READY
2000 submitted: C677IJNMHEQJD2ATRIRJUWJB READY
2001 submitted: FZHWYBNCB3XJCZYKDN57PKD4 READY
2002 submitted: I27YKQSHKLZWRX2LUVWOIUWQ READY
2003 submitted: 4VYPRBAH26OSGJJKRIS7FIE6 READY
2004 submitted: DC76NPIRMTARHUOKHVSZI5UY READY
2005 submitted: RKJM6E7FDAQ7D5CNXBGT2LBJ READY
2006 submitted: ABFZPUHN5MKETBBLOBW7IBDP READY
2007 submitted: ETYGHETLNKO4ORY3AOOIEGSY READY
2008 submitted: J75CM2F47F7OTX5L4QDBN4MP READY
2009 submitted: NT35WXM4MQJ6LGMNVDPHSAWH READY
2010 submitted: RIDXIKKCKQYSDUIZC2IKOPT6 READY
2011 submitted: STPJUU4R5SYNFMNAMV2MRL6V READY
2012 submitted: H65EMW7XXCRRAQHIQUMOJSWM READY
2013 submitted: NAMEWGOPB6R73SKNKCSKR55F READY
2014 submitted: KO7J6SKCM5TYNBK22GS3MQNX READY
2015 submitted: 2BFGDVFQKOS7YA7NDWNYQAGL READY
2016 submitted: AYGDAZECZIEWIBK3UXUGE3CN READY
2017 submitted: 47LOLWCAMUL5KRU542MHHELV READY
2018 submitted: 4PI4ZSHJNQV4JY37CD3JMVE2 READY
2019 submitte

In [ ]:
import shutil
import calendar

def expected_months(year):
    return list(range(1, 10)) if year == 2025 else list(range(1, 13))


def audit_year(path, year):
    df = pd.read_csv(path, dtype={"huc2": "string"})
    expected = {
        (code, year, month)
        for code in HUC2_CODES
        for month in expected_months(year)
    }

    actual = {
        (str(code), int(y), int(m))
        for code, y, m in df[["huc2", "year", "month"]].itertuples(
            index=False, name=None
        )
    }

    problems = []

    if len(df) != len(expected):
        problems.append(f"rows={len(df)}, expected={len(expected)}")
    if df.duplicated(["huc2", "year", "month"]).any():
        problems.append("duplicate basin-month rows")
    if actual != expected:
        problems.append(
            f"missing={len(expected - actual)}, extra={len(actual - expected)}"
        )

    value_cols = ["pr_mm_month", "eto_mm_month", "etr_mm_month"]
    if df[value_cols].isna().any().any():
        problems.append("missing monthly values")

    if not df["status_values"].eq("permanent").all():
        problems.append("non-permanent image status")

    if (df[value_cols] < 0).any().any():
        problems.append("negative monthly value")

    expected_days = df.apply(
        lambda r: calendar.monthrange(int(r["year"]), int(r["month"]))[1],
        axis=1
    )

    if not df["n_daily_images"].eq(expected_days).all():
        problems.append("daily image count differs from calendar days")

    for col in ["pr_pixel_count", "eto_pixel_count", "etr_pixel_count"]:
        if df[col].isna().any() or (df[col] <= 0).any():
            problems.append(f"invalid {col}")

    return df, problems


audit_log = []

for year in range(1998, 2026):
    filename = export_filename(year)
    source = EXPORT_DIR / filename
    destination = PROJECT_DIR / filename

    candidate = destination if destination.is_file() else source

    if not candidate.is_file():
        audit_log.append({
            "year": year, "status": "MISSING", "details": "CSV not visible"
        })
        continue

    try:
        df, problems = audit_year(candidate, year)

        if problems:
            audit_log.append({
                "year": year,
                "status": "CHECK_NEEDED",
                "details": "; ".join(problems)
            })
            continue

        if not destination.is_file():
            shutil.copy2(source, destination)

        audit_log.append({
            "year": year,
            "status": "VERIFIED",
            "details": f"{len(df)} rows"
        })

    except Exception as exc:
        audit_log.append({
            "year": year,
            "status": "ERROR",
            "details": f"{type(exc).__name__}: {exc}"
        })

audit = pd.DataFrame(audit_log)
print(audit.to_string(index=False))
print("\nVerified years:", (audit["status"] == "VERIFIED").sum(), "/ 28")

 year   status  details
 1998 VERIFIED 204 rows
 1999 VERIFIED 204 rows
 2000 VERIFIED 204 rows
 2001 VERIFIED 204 rows
 2002 VERIFIED 204 rows
 2003 VERIFIED 204 rows
 2004 VERIFIED 204 rows
 2005 VERIFIED 204 rows
 2006 VERIFIED 204 rows
 2007 VERIFIED 204 rows
 2008 VERIFIED 204 rows
 2009 VERIFIED 204 rows
 2010 VERIFIED 204 rows
 2011 VERIFIED 204 rows
 2012 VERIFIED 204 rows
 2013 VERIFIED 204 rows
 2014 VERIFIED 204 rows
 2015 VERIFIED 204 rows
 2016 VERIFIED 204 rows
 2017 VERIFIED 204 rows
 2018 VERIFIED 204 rows
 2019 VERIFIED 204 rows
 2020 VERIFIED 204 rows
 2021 VERIFIED 204 rows
 2022 VERIFIED 204 rows
 2023 VERIFIED 204 rows
 2024 VERIFIED 204 rows
 2025 VERIFIED 153 rows

Verified years: 28 / 28


In [ ]:
if not audit["status"].eq("VERIFIED").all():
    raise RuntimeError(
        "Some years are missing or failed audit. "
        "Resolve those years before combining."
    )

frames = []

for year in range(1998, 2026):
    path = PROJECT_DIR / export_filename(year)
    df, problems = audit_year(path, year)

    if problems:
        raise RuntimeError(f"{year}: {problems}")

    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(
    ["huc2", "year", "month"]
).reset_index(drop=True)

expected_rows = 17 * (27 * 12 + 9)

if len(combined) != expected_rows:
    raise RuntimeError(
        f"Combined rows: {len(combined)}; expected {expected_rows}"
    )

output = PROJECT_DIR / "gridmet_monthly_17huc02_1998_2025_09_fixedgrid.csv"
combined.to_csv(output, index=False)

print("Saved:", output)
print("Rows:", len(combined))
print("Basins:", combined["huc2"].nunique())
print("First month:", combined[["year", "month"]].iloc[0].to_dict())
print("Last month:", combined[["year", "month"]].iloc[-1].to_dict())

Saved: /content/drive/MyDrive/ecohydro_project/data/raw/gridmet/gridmet_monthly_17huc02_1998_2025_09_fixedgrid.csv
Rows: 5661
Basins: 17
First month: {'year': 1998, 'month': 1}
Last month: {'year': 2025, 'month': 9}


###IMERG-------------------------------------------------------


In [ ]:
import ee
import os

ee.Initialize(project="ecohydro-project")

PROJECT_RAW = "/content/drive/MyDrive/ecohydro_project/data/raw"
os.makedirs(PROJECT_RAW, exist_ok=True)

# Earth Engine accepts a Drive folder NAME, not a nested Colab path.
DRIVE_EXPORT_FOLDER = "ecohydro_imerg_hh_export"

test_basin = basins.filter(ee.Filter.eq("huc2", "10"))

imerg_day = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate("2020-01-01", "2020-01-02")
    .filter(ee.Filter.eq("status", "permanent"))
    .select("precipitation")
)

print("Permanent half-hourly images for test day:", imerg_day.size().getInfo())
print("Test basin count:", test_basin.size().getInfo())

NameError: name 'basins' is not defined

In [ ]:
def image_to_basin_rows(image):
    image = ee.Image(image)
    timestamp = ee.Date(image.get("system:time_start"))

    reduced = image.reduceRegions(
        collection=test_basin,
        reducer=ee.Reducer.mean(),
        scale=11132,
        tileScale=4
    )

    return reduced.map(
        lambda feature: ee.Feature(
            None,  # Do not attach the very large basin polygon to each row.
            {
                "huc2": feature.get("huc2"),
                "name": feature.get("name"),
                "timestamp_utc": timestamp.format("YYYY-MM-dd HH:mm:ss"),
                "precipitation_mm_per_hour": feature.get("precipitation"),
                "status": image.get("status")
            }
        )
    )

test_table = imerg_day.map(image_to_basin_rows).flatten()

task = ee.batch.Export.table.toDrive(
    collection=test_table,
    description="imerg_v07_hh_test_huc10_20200101",
    folder=DRIVE_EXPORT_FOLDER,
    fileNamePrefix="imerg_v07_hh_test_huc10_20200101",
    fileFormat="CSV",
    selectors=[
        "huc2",
        "name",
        "timestamp_utc",
        "precipitation_mm_per_hour",
        "status"
    ]
)

task.start()
print("Task ID:", task.id)
print("Task status:", task.status())

NameError: name 'imerg_day' is not defined

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil
import pandas as pd
import numpy as np
import ee

drive.mount("/content/drive")
ee.Initialize(project="ecohydro-project")

source = Path(
    "/content/drive/MyDrive/ecohydro_imerg_hh_export/"
    "imerg_v07_hh_test_huc10_20200101.csv"
)

destination_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly"
)
destination_dir.mkdir(parents=True, exist_ok=True)
destination = destination_dir / source.name

if not source.is_file():
    raise FileNotFoundError(
        f"Completed task's CSV is not visible at the expected location: {source}"
    )

if not destination.exists():
    shutil.copy2(source, destination)
    print("Copied to:", destination)
else:
    print("Already present; not overwritten:", destination)

df = pd.read_csv(destination, dtype={"huc2": "string"})

required = [
    "huc2", "name", "timestamp_utc",
    "precipitation_mm_per_hour", "status"
]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df["timestamp_utc"] = pd.to_datetime(
    df["timestamp_utc"], errors="coerce", utc=True
)
df["precipitation_mm_per_hour"] = pd.to_numeric(
    df["precipitation_mm_per_hour"], errors="coerce"
)

expected_times = pd.date_range(
    "2020-01-01 00:00:00",
    periods=48,
    freq="30min",
    tz="UTC"
)
actual_times = pd.DatetimeIndex(df["timestamp_utc"].dropna())

missing_times = expected_times.difference(actual_times)
extra_times = actual_times.difference(expected_times)

print("\n=== FILE CHECK ===")
print("Rows:", len(df), "(expected 48)")
print("Basin IDs:", sorted(df["huc2"].dropna().unique().tolist()),
      "(expected ['10'])")
print("Unique timestamps:", df["timestamp_utc"].nunique(), "(expected 48)")
print("Earliest:", df["timestamp_utc"].min())
print("Latest:", df["timestamp_utc"].max())
print("Missing expected timestamps:", len(missing_times))
print("Extra timestamps:", len(extra_times))
print("Duplicate basin-timestamp rows:",
      df.duplicated(["huc2", "timestamp_utc"]).sum())
print("Missing/invalid precipitation:",
      df["precipitation_mm_per_hour"].isna().sum())
print("Negative precipitation:",
      int((df["precipitation_mm_per_hour"] < 0).sum()))
print("Statuses:", df["status"].value_counts(dropna=False).to_dict())
print("Rate range (mm/hour):",
      df["precipitation_mm_per_hour"].min(),
      "to",
      df["precipitation_mm_per_hour"].max())

passed = (
    len(df) == 48
    and set(df["huc2"].dropna()) == {"10"}
    and df["timestamp_utc"].nunique() == 48
    and len(missing_times) == 0
    and len(extra_times) == 0
    and df.duplicated(["huc2", "timestamp_utc"]).sum() == 0
    and df["precipitation_mm_per_hour"].notna().all()
    and (df["precipitation_mm_per_hour"] >= 0).all()
    and df["status"].eq("permanent").all()
)

print("\nONE-DAY CSV VERDICT:", "PASS" if passed else "CHECK NEEDED")

imerg_test = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate("2020-01-01", "2020-01-02")
    .filter(ee.Filter.eq("status", "permanent"))
    .select("precipitation")
)
native_projection = ee.Image(
    imerg_test.first()
).select("precipitation").projection()

print("\n=== EARTH ENGINE SOURCE CHECK ===")
print("Permanent images on test day:", imerg_test.size().getInfo())
print("Native precipitation projection:")
print(native_projection.getInfo())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already present; not overwritten: /content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly/imerg_v07_hh_test_huc10_20200101.csv

=== FILE CHECK ===
Rows: 48 (expected 48)
Basin IDs: ['10'] (expected ['10'])
Unique timestamps: 48 (expected 48)
Earliest: 2020-01-01 00:00:00+00:00
Latest: 2020-01-01 23:30:00+00:00
Missing expected timestamps: 0
Extra timestamps: 0
Duplicate basin-timestamp rows: 0
Missing/invalid precipitation: 48
Negative precipitation: 0
Statuses: {'permanent': 48}
Rate range (mm/hour): nan to nan

ONE-DAY CSV VERDICT: CHECK NEEDED

=== EARTH ENGINE SOURCE CHECK ===


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Permanent images on test day: 48
Native precipitation projection:
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.1, 0, -180, 0, -0.1, 90]}


In [ ]:
import ee

ee.Initialize(project="ecohydro-project")

basin = (
    ee.FeatureCollection("USGS/WBD/2017/HUC02")
    .filter(ee.Filter.eq("huc2", "10"))
    .select(["huc2", "name"])
)

collection = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate("2020-01-01", "2020-01-02")
    .filter(ee.Filter.eq("status", "permanent"))
    .select("precipitation")
)

image = ee.Image(collection.first())
native_projection = image.select("precipitation").projection()

print("Images in test day:", collection.size().getInfo())
print("Basin count:", basin.size().getInfo())
print("Image band names:", image.bandNames().getInfo())
print("Image native projection:", native_projection.getInfo())

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(),
    sharedInputs=True
)

result = image.reduceRegions(
    collection=basin,
    reducer=reducer,
    crs=native_projection,
    tileScale=4
)

feature = ee.Feature(result.first())

print("\nFULL REDUCTION PROPERTIES:")
print(feature.toDictionary().getInfo())

print("\nEXPECTED RESULT FIELDS:")
print("precipitation_mean:",
      feature.get("precipitation_mean").getInfo())
print("precipitation_count:",
      feature.get("precipitation_count").getInfo())

Images in test day: 48
Basin count: 1
Image band names: ['precipitation']
Image native projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.1, 0, -180, 0, -0.1, 90]}

FULL REDUCTION PROPERTIES:
{'count': 15540, 'huc2': '10', 'mean': 0.02196356547906548, 'name': 'Missouri Region'}

EXPECTED RESULT FIELDS:


EEException: Collection.first: Error in map(ID=null):
Feature.select: Parameter 'input' is required and may not be null.

In [ ]:
import ee

ee.Initialize(project="ecohydro-project")

DRIVE_EXPORT_FOLDER = "ecohydro_imerg_hh_export"

test_basin = (
    ee.FeatureCollection("USGS/WBD/2017/HUC02")
    .filter(ee.Filter.eq("huc2", "10"))
    .select(["huc2", "name"])
)

imerg_day = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate("2020-01-01", "2020-01-02")
    .filter(ee.Filter.eq("status", "permanent"))
    .select("precipitation")
)

native_projection = (
    ee.Image(imerg_day.first())
    .select("precipitation")
    .projection()
)

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(),
    sharedInputs=True
)

def image_to_basin_row_corrected(image):
    image = ee.Image(image)
    timestamp = ee.Date(image.get("system:time_start"))

    reduced = image.reduceRegions(
        collection=test_basin,
        reducer=reducer,
        crs=native_projection,
        tileScale=4
    )

    return reduced.map(
        lambda feature: ee.Feature(None, {
            "huc2": feature.get("huc2"),
            "name": feature.get("name"),
            "timestamp_utc": timestamp.format("YYYY-MM-dd HH:mm:ss"),
            "precipitation_mm_per_hour": feature.get("mean"),
            "valid_pixel_count": feature.get("count"),
            "status": image.get("status"),
            "product": "NASA/GPM_L3/IMERG_V07"
        })
    )

corrected_table = imerg_day.map(
    image_to_basin_row_corrected
).flatten()

task_corrected = ee.batch.Export.table.toDrive(
    collection=corrected_table,
    description="imerg_v07_hh_test_huc10_20200101_fixedgrid",
    folder=DRIVE_EXPORT_FOLDER,
    fileNamePrefix="imerg_v07_hh_test_huc10_20200101_fixedgrid",
    fileFormat="CSV",
    selectors=[
        "huc2", "name", "timestamp_utc",
        "precipitation_mm_per_hour", "valid_pixel_count",
        "status", "product"
    ]
)

task_corrected.start()

print("Corrected task ID:", task_corrected.id)
print("Current state:", task_corrected.status()["state"])

Corrected task ID: RSX7NLY256Z6KVCIX65CIBN3
Current state: READY


In [ ]:
from pathlib import Path
from google.colab import drive
import shutil
import pandas as pd

drive.mount("/content/drive")

source = Path(
    "/content/drive/MyDrive/ecohydro_imerg_hh_export/"
    "imerg_v07_hh_test_huc10_20200101_fixedgrid.csv"
)

destination_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly"
)
destination_dir.mkdir(parents=True, exist_ok=True)
destination = destination_dir / source.name

if not source.is_file():
    raise FileNotFoundError(f"Corrected CSV not found: {source}")

if not destination.exists():
    shutil.copy2(source, destination)
    print("Copied:", destination)
else:
    print("Already present; not overwritten:", destination)

df = pd.read_csv(destination, dtype={"huc2": "string"})

df["timestamp_utc"] = pd.to_datetime(
    df["timestamp_utc"], errors="coerce", utc=True
)
df["precipitation_mm_per_hour"] = pd.to_numeric(
    df["precipitation_mm_per_hour"], errors="coerce"
)
df["valid_pixel_count"] = pd.to_numeric(
    df["valid_pixel_count"], errors="coerce"
)

expected = pd.date_range(
    "2020-01-01 00:00:00",
    periods=48,
    freq="30min",
    tz="UTC"
)

actual = pd.DatetimeIndex(df["timestamp_utc"].dropna())
missing = expected.difference(actual)
extra = actual.difference(expected)

print("Rows:", len(df), "(expect 48)")
print("Basins:", df["huc2"].unique().tolist(), "(expect ['10'])")
print("Unique timestamps:", df["timestamp_utc"].nunique(), "(expect 48)")
print("Missing timestamps:", len(missing))
print("Extra timestamps:", len(extra))
print("Duplicate basin-timestamp rows:",
      df.duplicated(["huc2", "timestamp_utc"]).sum())
print("Missing rainfall values:",
      df["precipitation_mm_per_hour"].isna().sum())
print("Negative rainfall values:",
      int((df["precipitation_mm_per_hour"] < 0).sum()))
print("Missing/zero valid-pixel counts:",
      int((df["valid_pixel_count"].isna() |
           (df["valid_pixel_count"] <= 0)).sum()))
print("Status:", df["status"].value_counts(dropna=False).to_dict())
print("Rainfall rate range (mm/hour):",
      df["precipitation_mm_per_hour"].min(),
      "to",
      df["precipitation_mm_per_hour"].max())

passed = (
    len(df) == 48
    and set(df["huc2"].dropna()) == {"10"}
    and df["timestamp_utc"].nunique() == 48
    and len(missing) == 0
    and len(extra) == 0
    and df.duplicated(["huc2", "timestamp_utc"]).sum() == 0
    and df["precipitation_mm_per_hour"].notna().all()
    and (df["precipitation_mm_per_hour"] >= 0).all()
    and df["valid_pixel_count"].gt(0).all()
    and df["status"].eq("permanent").all()
)

print("CORRECTED ONE-DAY TEST:", "PASS" if passed else "CHECK NEEDED")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already present; not overwritten: /content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly/imerg_v07_hh_test_huc10_20200101_fixedgrid.csv
Rows: 48 (expect 48)
Basins: ['10'] (expect ['10'])
Unique timestamps: 48 (expect 48)
Missing timestamps: 0
Extra timestamps: 0
Duplicate basin-timestamp rows: 0
Missing rainfall values: 0
Negative rainfall values: 0
Missing/zero valid-pixel counts: 0
Status: {'permanent': 48}
Rainfall rate range (mm/hour): 0.0173079035176158 to 0.0983586558173376
CORRECTED ONE-DAY TEST: PASS
  huc2             name             timestamp_utc  precipitation_mm_per_hour  \
0   10  Missouri Region 2020-01-01 00:00:00+00:00                   0.021964   
1   10  Missouri Region 2020-01-01 00:30:00+00:00                   0.019012   
2   10  Missouri Region 2020-01-01 01:00:00+00:00                   0.017308   
3   10  Missouri Region 

In [ ]:
import ee
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ee.Initialize(project="ecohydro-project")

HUC2_CODES = [
    "01", "02", "03", "04", "05", "07", "08", "09", "10",
    "11", "12", "13", "14", "15", "16", "17", "18"
]

basins = (
    ee.FeatureCollection("USGS/WBD/2017/HUC02")
    .filter(ee.Filter.inList("huc2", HUC2_CODES))
    .select(["huc2", "name"])
)

start = ee.Date("2020-01-01")
end = ee.Date("2020-02-01")

imerg_month = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate(start, end)
    .filter(ee.Filter.eq("status", "permanent"))
    .select("precipitation")
)

native_projection = (
    ee.Image(imerg_month.first())
    .select("precipitation")
    .projection()
)

n_images = imerg_month.size().getInfo()
n_basins = basins.size().getInfo()

print("Permanent images:", n_images, "(expect 1488)")
print("Basins:", n_basins, "(expect 17)")
print("Native projection:", native_projection.getInfo())

if n_images != 1488 or n_basins != 17:
    raise RuntimeError(
        "Unexpected number of permanent images or basins. "
        "Do not submit the export."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Permanent images: 1488 (expect 1488)
Basins: 17 (expect 17)
Native projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.1, 0, -180, 0, -0.1, 90]}


In [ ]:
DRIVE_EXPORT_FOLDER = "ecohydro_imerg_hh_export"
FILE_PREFIX = "imerg_v07_hh_17huc02_2020_01_fixedgrid"

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(),
    sharedInputs=True
)

def image_to_basin_rows(image):
    image = ee.Image(image)
    timestamp = ee.Date(image.get("system:time_start"))

    reduced = image.reduceRegions(
        collection=basins,
        reducer=reducer,
        crs=native_projection,
        tileScale=4
    )

    return reduced.map(
        lambda feature: ee.Feature(None, {
            "huc2": feature.get("huc2"),
            "name": feature.get("name"),
            "timestamp_utc": timestamp.format("YYYY-MM-dd HH:mm:ss"),
            "precipitation_mm_per_hour": feature.get("mean"),
            "valid_pixel_count": feature.get("count"),
            "status": image.get("status"),
            "product": "NASA/GPM_L3/IMERG_V07"
        })
    )

month_table = imerg_month.map(image_to_basin_rows).flatten()

task_month = ee.batch.Export.table.toDrive(
    collection=month_table,
    description=FILE_PREFIX,
    folder=DRIVE_EXPORT_FOLDER,
    fileNamePrefix=FILE_PREFIX,
    fileFormat="CSV",
    selectors=[
        "huc2", "name", "timestamp_utc",
        "precipitation_mm_per_hour", "valid_pixel_count",
        "status", "product"
    ]
)

task_month.start()
print("Submitted task ID:", task_month.id)
print("Initial task state:", task_month.status()["state"])

Submitted task ID: 6GI7233JNK6EM26NFIAJUO3E
Initial task state: READY


In [ ]:
from pathlib import Path
import shutil
import pandas as pd

filename = "imerg_v07_hh_17huc02_2020_01_fixedgrid.csv"

source = (
    Path("/content/drive/MyDrive/ecohydro_imerg_hh_export")
    / filename
)
project_dir = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly"
)
project_dir.mkdir(parents=True, exist_ok=True)

destination = project_dir / filename

if not source.is_file():
    raise FileNotFoundError(
        f"Completed export not found at expected location: {source}"
    )

# Inspect the exported source before copying it into the project.
df = pd.read_csv(source, dtype={"huc2": "string"})

required_columns = {
    "huc2", "name", "timestamp_utc",
    "precipitation_mm_per_hour", "valid_pixel_count",
    "status", "product"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing CSV columns: {sorted(missing_columns)}")

df["timestamp_utc"] = pd.to_datetime(
    df["timestamp_utc"], errors="coerce", utc=True
)
df["precipitation_mm_per_hour"] = pd.to_numeric(
    df["precipitation_mm_per_hour"], errors="coerce"
)
df["valid_pixel_count"] = pd.to_numeric(
    df["valid_pixel_count"], errors="coerce"
)

expected_times = pd.date_range(
    "2020-01-01 00:00:00",
    "2020-01-31 23:30:00",
    freq="30min",
    tz="UTC"
)
expected_basins = set(HUC2_CODES)

issues = []

if len(df) != 17 * 1488:
    issues.append(f"rows={len(df)}, expected={17 * 1488}")

if set(df["huc2"].dropna()) != expected_basins:
    issues.append("basin IDs do not match the expected 17")

if df["timestamp_utc"].isna().any():
    issues.append("unparseable timestamps")

if df.duplicated(["huc2", "timestamp_utc"]).any():
    issues.append("duplicate basin–timestamp combinations")

for code in HUC2_CODES:
    actual = pd.DatetimeIndex(
        df.loc[df["huc2"] == code, "timestamp_utc"].dropna()
    )
    missing = expected_times.difference(actual)
    extra = actual.difference(expected_times)

    if len(missing) or len(extra):
        issues.append(
            f"HUC02 {code}: {len(missing)} missing, "
            f"{len(extra)} extra timestamps"
        )

if df["precipitation_mm_per_hour"].isna().any():
    issues.append("missing/invalid rainfall rates")

if (df["precipitation_mm_per_hour"] < 0).any():
    issues.append("negative rainfall rates")

if df["valid_pixel_count"].isna().any():
    issues.append("missing pixel counts")

if (df["valid_pixel_count"] <= 0).any():
    issues.append("zero/negative pixel counts")

if not df["status"].eq("permanent").all():
    issues.append("non-permanent image status")

if not df["product"].eq("NASA/GPM_L3/IMERG_V07").all():
    issues.append("unexpected product identifier")

print("Rows:", len(df), "(expected 25,296)")
print("Basins:", df["huc2"].nunique(), "(expected 17)")
print("Distinct timestamps:", df["timestamp_utc"].nunique(),
      "(expected 1,488)")
print("Duplicate basin–timestamp pairs:",
      df.duplicated(["huc2", "timestamp_utc"]).sum())
print("Missing rainfall values:",
      df["precipitation_mm_per_hour"].isna().sum())
print("Invalid pixel counts:",
      (df["valid_pixel_count"].isna() |
       (df["valid_pixel_count"] <= 0)).sum())
print("Statuses:", df["status"].value_counts(dropna=False).to_dict())
print("Issues:", issues)

if issues:
    print("MONTH-LONG TEST: CHECK NEEDED")
else:
    if destination.exists():
        print("Validated, but project copy already exists; not overwritten:",
              destination)
    else:
        shutil.copy2(source, destination)
        print("Validated and copied to:", destination)
    print("MONTH-LONG TEST: PASS")

NameError: name 'HUC2_CODES' is not defined

In [ ]:
import ee
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ee.Initialize(project="ecohydro-project")

HUC2_CODES = [
    "01", "02", "03", "04", "05", "07", "08", "09", "10",
    "11", "12", "13", "14", "15", "16", "17", "18"
]

EXPORT_FOLDER_NAME = "ecohydro_imerg_hh_export"
EXPORT_DIR = Path("/content/drive/MyDrive") / EXPORT_FOLDER_NAME
PROJECT_DIR = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly"
)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PROJECT_DIR / "imerg_hh_export_manifest.csv"

basins = (
    ee.FeatureCollection("USGS/WBD/2017/HUC02")
    .filter(ee.Filter.inList("huc2", HUC2_CODES))
    .select(["huc2", "name"])
)

imerg = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .select("precipitation")
)

native_projection = (
    ee.Image(
        imerg.filterDate("2020-01-01", "2020-01-02").first()
    )
    .select("precipitation")
    .projection()
)

reducer = ee.Reducer.mean().combine(
    reducer2=ee.Reducer.count(),
    sharedInputs=True
)

assert basins.size().getInfo() == 17
print("17 basins ready.")
print("Native projection:", native_projection.getInfo())
print("Project folder:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


17 basins ready.
Native projection: {'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.1, 0, -180, 0, -0.1, 90]}
Project folder: /content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly


In [ ]:
def month_key(year, month):
    return f"{year:04d}-{month:02d}"


def filename_for_month(year, month):
    return (
        f"imerg_v07_hh_17huc02_"
        f"{year:04d}_{month:02d}_fixedgrid.csv"
    )


def expected_images(year, month):
    return 48 * pd.Period(f"{year:04d}-{month:02d}").days_in_month


def build_month_collection(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    return (
        imerg
        .filterDate(start, end)
        .filter(ee.Filter.eq("status", "permanent"))
    )


def image_to_basin_rows(image):
    image = ee.Image(image)
    timestamp = ee.Date(image.get("system:time_start"))

    reduced = image.reduceRegions(
        collection=basins,
        reducer=reducer,
        crs=native_projection,
        tileScale=4
    )

    return reduced.map(
        lambda feature: ee.Feature(None, {
            "huc2": feature.get("huc2"),
            "name": feature.get("name"),
            "timestamp_utc": timestamp.format("YYYY-MM-dd HH:mm:ss"),
            "precipitation_mm_per_hour": feature.get("mean"),
            "valid_pixel_count": feature.get("count"),
            "status": image.get("status"),
            "product": "NASA/GPM_L3/IMERG_V07"
        })
    )

In [ ]:
def load_manifest():
    if MANIFEST_PATH.is_file():
        manifest = pd.read_csv(
            MANIFEST_PATH,
            dtype={"month_key": "string", "task_id": "string"}
        )
        return manifest

    return pd.DataFrame(columns=[
        "month_key", "year", "month", "filename",
        "task_id", "state", "note"
    ])


def save_manifest(manifest):
    manifest.to_csv(MANIFEST_PATH, index=False)


def months_to_process():
    return [
        (year, month)
        for year in range(1998, 2026)
        for month in range(1, 10 if year == 2025 else 13)
    ]


def submit_next_batch(batch_size=5):
    manifest = load_manifest()
    recorded = set(manifest["month_key"].dropna().astype(str))
    submitted_now = []

    for year, month in months_to_process():
        if len(submitted_now) >= batch_size:
            break

        key = month_key(year, month)
        filename = filename_for_month(year, month)

        if (PROJECT_DIR / filename).is_file():
            print(key, "already in project folder; skipped")
            continue

        if (EXPORT_DIR / filename).is_file():
            print(key, "already in Earth Engine export folder; skipped")
            continue

        if key in recorded:
            print(key, "already in manifest; skipped")
            continue

        collection = build_month_collection(year, month)
        count = collection.size().getInfo()
        expected = expected_images(year, month)

        if count != expected:
            print(
                key, f"NOT SUBMITTED: permanent images={count}, "
                f"expected={expected}"
            )
            manifest.loc[len(manifest)] = [
                key, year, month, filename, "",
                "SOURCE_CHECK_FAILED",
                f"permanent images={count}; expected={expected}"
            ]
            save_manifest(manifest)
            continue

        table = collection.map(image_to_basin_rows).flatten()
        prefix = filename.removesuffix(".csv")

        task = ee.batch.Export.table.toDrive(
            collection=table,
            description=prefix,
            folder=EXPORT_FOLDER_NAME,
            fileNamePrefix=prefix,
            fileFormat="CSV",
            selectors=[
                "huc2", "name", "timestamp_utc",
                "precipitation_mm_per_hour", "valid_pixel_count",
                "status", "product"
            ]
        )

        task.start()

        manifest.loc[len(manifest)] = [
            key, year, month, filename, task.id,
            task.status()["state"], ""
        ]
        save_manifest(manifest)

        submitted_now.append(key)
        print(key, "submitted:", task.id)

    print("Submitted in this batch:", submitted_now)
    return manifest

In [ ]:
manifest = submit_next_batch(batch_size=5)

1998-01 submitted: TTAVEO6FB52G4BAESJASVG47
1998-02 submitted: ZKLJIMSZVLAPYCDT2GJECOYJ
1998-03 submitted: ENAUXTWNBUSTCVQ3IBDGSS5D
1998-04 submitted: FQSXUYTCHU6JZHBRFSGN7DZQ
1998-05 submitted: N5YDOEEQZDXBG7JQTFZYIWYB
Submitted in this batch: ['1998-01', '1998-02', '1998-03', '1998-04', '1998-05']


In [ ]:
import shutil

REQUIRED_COLUMNS = {
    "huc2", "name", "timestamp_utc",
    "precipitation_mm_per_hour", "valid_pixel_count",
    "status", "product"
}


def audit_month_file(path, year, month):
    df = pd.read_csv(path, dtype={"huc2": "string"})

    missing_columns = REQUIRED_COLUMNS - set(df.columns)
    if missing_columns:
        return [f"missing columns: {sorted(missing_columns)}"]

    timestamps = pd.to_datetime(
        df["timestamp_utc"], errors="coerce", utc=True
    )
    rain = pd.to_numeric(
        df["precipitation_mm_per_hour"], errors="coerce"
    )
    pixel_count = pd.to_numeric(
        df["valid_pixel_count"], errors="coerce"
    )

    start = pd.Timestamp(
        year=year, month=month, day=1, tz="UTC"
    )
    expected_times = pd.date_range(
        start=start,
        periods=expected_images(year, month),
        freq="30min"
    )

    issues = []

    if len(df) != len(HUC2_CODES) * len(expected_times):
        issues.append(
            f"rows={len(df)}; expected="
            f"{len(HUC2_CODES) * len(expected_times)}"
        )

    if timestamps.isna().any():
        issues.append("unparseable timestamps")

    if df.assign(parsed_time=timestamps).duplicated(
        ["huc2", "parsed_time"]
    ).any():
        issues.append("duplicate basin-timestamp rows")

    if set(df["huc2"].dropna()) != set(HUC2_CODES):
        issues.append("basin IDs do not match the 17 expected IDs")

    for code in HUC2_CODES:
        actual = pd.DatetimeIndex(
            timestamps.loc[df["huc2"] == code].dropna()
        )
        missing = expected_times.difference(actual)
        extra = actual.difference(expected_times)

        if len(missing) or len(extra):
            issues.append(
                f"{code}: {len(missing)} missing, "
                f"{len(extra)} extra timestamps"
            )

    if rain.isna().any():
        issues.append(f"{rain.isna().sum()} missing rainfall values")

    if (rain < 0).any():
        issues.append(f"{(rain < 0).sum()} negative rainfall values")

    if pixel_count.isna().any() or (pixel_count <= 0).any():
        issues.append("missing or nonpositive valid-pixel counts")

    if not df["status"].eq("permanent").all():
        issues.append("non-permanent images present")

    if not df["product"].eq(
        "NASA/GPM_L3/IMERG_V07"
    ).all():
        issues.append("unexpected product identifier")

    return issues


def audit_available_exports():
    manifest = load_manifest()
    results = []

    for year, month in months_to_process():
        key = month_key(year, month)
        filename = filename_for_month(year, month)
        source = EXPORT_DIR / filename
        destination = PROJECT_DIR / filename

        if destination.is_file():
            candidate = destination
        elif source.is_file():
            candidate = source
        else:
            continue

        try:
            issues = audit_month_file(candidate, year, month)

            if issues:
                results.append({
                    "month": key,
                    "result": "CHECK_NEEDED",
                    "details": "; ".join(issues)
                })
                continue

            if not destination.is_file():
                shutil.copy2(source, destination)

            results.append({
                "month": key,
                "result": "VERIFIED",
                "details": f"{expected_images(year, month) * 17} rows"
            })

        except Exception as exc:
            results.append({
                "month": key,
                "result": "ERROR",
                "details": f"{type(exc).__name__}: {exc}"
            })

    report = pd.DataFrame(
        results, columns=["month", "result", "details"]
    )
    print(report.to_string(index=False))
    print(
        "Verified month files:",
        int(report["result"].eq("VERIFIED").sum()),
        "/ 333"
    )
    return report

In [ ]:
report = audit_available_exports()

  month   result    details
1998-01 VERIFIED 25296 rows
1998-02 VERIFIED 22848 rows
1998-03 VERIFIED 25296 rows
1998-04 VERIFIED 24480 rows
1998-05 VERIFIED 25296 rows
1998-06 VERIFIED 24480 rows
1998-07 VERIFIED 25296 rows
1998-08 VERIFIED 25296 rows
1998-09 VERIFIED 24480 rows
1998-10 VERIFIED 25296 rows
1998-11 VERIFIED 24480 rows
1998-12 VERIFIED 25296 rows
1999-01 VERIFIED 25296 rows
2020-01 VERIFIED 25296 rows
Verified month files: 14 / 333


In [ ]:
def refresh_manifest():
    manifest = load_manifest()
    if manifest.empty:
        print("Manifest is empty.")
        return manifest

    tasks = {t.id: t.status() for t in ee.batch.Task.list()}

    for idx, row in manifest.iterrows():
        task_id = str(row["task_id"]) if pd.notna(row["task_id"]) else ""
        if task_id and task_id in tasks:
            status = tasks[task_id]
            manifest.at[idx, "state"] = status.get("state", "")
            err = status.get("error_message", "")
            if err:
                manifest.at[idx, "note"] = str(err)[:300]

    save_manifest(manifest)
    print(manifest["state"].value_counts(dropna=False).to_string())
    return manifest

manifest = refresh_manifest()

NameError: name 'MANIFEST_PATH' is not defined

In [ ]:
ACTIVE_STATES = {"READY", "RUNNING"}
FAILED_STATES = {"FAILED", "CANCELLED", "CANCEL_REQUESTED"}

def submit_queue_aware(target_active=8, max_new=10):
    manifest = refresh_manifest()
    active = int(manifest["state"].isin(ACTIVE_STATES).sum())
    recorded = set(manifest["month_key"].dropna().astype(str))
    submitted_now = []

    for year, month in months_to_process():
        if active >= target_active or len(submitted_now) >= max_new:
            break

        key = month_key(year, month)
        filename = filename_for_month(year, month)

        if (PROJECT_DIR / filename).is_file() or (EXPORT_DIR / filename).is_file():
            continue
        if key in recorded:
            continue

        collection = build_month_collection(year, month)
        count = collection.size().getInfo()
        expected = expected_images(year, month)

        if count != expected:
            print(key, f"NOT SUBMITTED: permanent images={count}, expected={expected}")
            manifest.loc[len(manifest)] = [
                key, year, month, filename, "", "SOURCE_CHECK_FAILED",
                f"permanent images={count}; expected={expected}"
            ]
            save_manifest(manifest)
            recorded.add(key)
            continue

        table = collection.map(image_to_basin_rows).flatten()
        prefix = filename.removesuffix(".csv")

        task = ee.batch.Export.table.toDrive(
            collection=table,
            description=prefix,
            folder=EXPORT_FOLDER_NAME,
            fileNamePrefix=prefix,
            fileFormat="CSV",
            selectors=[
                "huc2", "name", "timestamp_utc",
                "precipitation_mm_per_hour", "valid_pixel_count",
                "status", "product"
            ]
        )
        task.start()

        manifest.loc[len(manifest)] = [
            key, year, month, filename, task.id, task.status()["state"], ""
        ]
        save_manifest(manifest)
        recorded.add(key)
        active += 1
        submitted_now.append(key)
        print(key, "submitted:", task.id)

    failed = manifest[manifest["state"].isin(FAILED_STATES | {"SOURCE_CHECK_FAILED"})]
    print("\nActive tasks:", active, "| submitted this pass:", len(submitted_now))
    if len(failed):
        print("NEEDS ATTENTION:")
        print(failed[["month_key", "state", "note"]].to_string(index=False))
    return manifest

In [ ]:

manifest = submit_queue_aware(target_active=8, max_new=8)

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


state
COMPLETED    5


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


1998-06 submitted: BZTS6POX3DWIT5O6NUGMTS2U


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


...

1998-11 submitted: 5BDFZ2PV26I4VF3WJEKIPG44


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


1998-12 submitted: R533TBSDVGDC6VCOVIECPTPZ


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


1999-01 submitted: RKHPQ5QYNTESJXXX2B4N7AKA

Active tasks: 8 | submitted this pass: 8


In [ ]:
rows = []
for t in ee.batch.Task.list():
    s = t.status()
    if s.get("state") == "COMPLETED" and "imerg_v07_hh" in s.get("description", ""):
        rows.append({
            "task": s["description"],
            "eecu_seconds": s.get("batch_eecu_usage_seconds"),
        })

usage = pd.DataFrame(rows).dropna()
print(usage.to_string(index=False))

per_month = usage["eecu_seconds"].mean()
projected_hours = per_month * 333 / 3600
print(f"\nMean EECU-seconds per month: {per_month:,.0f}")
print(f"Projected for all 333 months: {projected_hours:,.1f} EECU-hours "
      f"(Community quota: 150; Contributor: 1,000)")

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


                                      task  eecu_seconds
    imerg_v07_hh_17huc02_1999_01_fixedgrid 171386.968750
    imerg_v07_hh_17huc02_1998_12_fixedgrid 152685.687500
    imerg_v07_hh_17huc02_1998_11_fixedgrid 106415.359375
    imerg_v07_hh_17huc02_1998_10_fixedgrid 152646.140625
    imerg_v07_hh_17huc02_1998_09_fixedgrid 181950.781250
    imerg_v07_hh_17huc02_1998_08_fixedgrid 205874.843750
    imerg_v07_hh_17huc02_1998_07_fixedgrid 157701.125000
    imerg_v07_hh_17huc02_1998_06_fixedgrid  36077.054688
    imerg_v07_hh_17huc02_1998_05_fixedgrid 160133.562500
    imerg_v07_hh_17huc02_1998_04_fixedgrid 141648.781250
    imerg_v07_hh_17huc02_1998_03_fixedgrid 127141.781250
    imerg_v07_hh_17huc02_1998_02_fixedgrid  98320.812500
    imerg_v07_hh_17huc02_1998_01_fixedgrid 174477.406250
    imerg_v07_hh_17huc02_2020_01_fixedgrid 165335.750000
imerg_v07_hh_test_huc10_20200101_fixedgrid    423.598816
          imerg_v07_hh_test_huc10_20200101    710.012329

Mean EECU-seconds per month: 1

In [ ]:
manifest = refresh_manifest()

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


state
COMPLETED    13


In [ ]:
VERIFIED_LOG = PROJECT_DIR / "imerg_hh_verified_months.csv"

def audit_new_exports():
    verified = (set(pd.read_csv(VERIFIED_LOG, dtype=str)["month"])
                if VERIFIED_LOG.is_file() else set())
    newly = []

    for year, month in months_to_process():
        key = month_key(year, month)
        if key in verified:
            continue

        filename = filename_for_month(year, month)
        source = EXPORT_DIR / filename
        destination = PROJECT_DIR / filename
        candidate = destination if destination.is_file() else source
        if not candidate.is_file():
            continue

        try:
            issues = audit_month_file(candidate, year, month)
        except Exception as exc:
            print(key, "ERROR:", type(exc).__name__, exc)
            continue

        if issues:
            print(key, "CHECK_NEEDED:", "; ".join(issues))
            continue

        if not destination.is_file():
            shutil.copy2(source, destination)
        verified.add(key)
        newly.append(key)
        print(key, "VERIFIED")

    pd.DataFrame({"month": sorted(verified)}).to_csv(VERIFIED_LOG, index=False)
    print(f"\nNewly verified: {len(newly)} | Total verified: {len(verified)} / 333")
    return sorted(verified)

verified_months = audit_new_exports()

NameError: name 'PROJECT_DIR' is not defined

In [ ]:
verified_months = audit_new_exports()


Newly verified: 0 | Total verified: 14 / 333


In [ ]:
rows = []
for t in ee.batch.Task.list():
    s = t.status()
    if s.get("state") == "COMPLETED" and "imerg_v07_hh" in s.get("description", ""):
        rows.append({
            "task": s["description"],
            "eecu_seconds": s.get("batch_eecu_usage_seconds"),
        })

usage = pd.DataFrame(rows).dropna()
per_month = usage["eecu_seconds"].mean()
print(usage.to_string(index=False))
print(f"\nMean EECU-seconds per month: {per_month:,.0f}")
print(f"Projected for all 333 months: {per_month * 333 / 3600:,.1f} EECU-hours")

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


                                      task  eecu_seconds
    imerg_v07_hh_17huc02_1999_01_fixedgrid 171386.968750
    imerg_v07_hh_17huc02_1998_12_fixedgrid 152685.687500
    imerg_v07_hh_17huc02_1998_11_fixedgrid 106415.359375
    imerg_v07_hh_17huc02_1998_10_fixedgrid 152646.140625
    imerg_v07_hh_17huc02_1998_09_fixedgrid 181950.781250
    imerg_v07_hh_17huc02_1998_08_fixedgrid 205874.843750
    imerg_v07_hh_17huc02_1998_07_fixedgrid 157701.125000
    imerg_v07_hh_17huc02_1998_06_fixedgrid  36077.054688
    imerg_v07_hh_17huc02_1998_05_fixedgrid 160133.562500
    imerg_v07_hh_17huc02_1998_04_fixedgrid 141648.781250
    imerg_v07_hh_17huc02_1998_03_fixedgrid 127141.781250
    imerg_v07_hh_17huc02_1998_02_fixedgrid  98320.812500
    imerg_v07_hh_17huc02_1998_01_fixedgrid 174477.406250
    imerg_v07_hh_17huc02_2020_01_fixedgrid 165335.750000
imerg_v07_hh_test_huc10_20200101_fixedgrid    423.598816
          imerg_v07_hh_test_huc10_20200101    710.012329

Mean EECU-seconds per month: 1

In [ ]:
manifest = submit_queue_aware(target_active=8, max_new=8)

NameError: name 'refresh_manifest' is not defined

In [ ]:
failed_task_id = "4ZUJKJ5Y6FHJ5X3L5MYPD5TE"

failed_status = ee.data.getTaskStatus(failed_task_id)[0]

print("Task:", failed_status.get("description"))
print("State:", failed_status.get("state"))
print("Error:", failed_status.get("error_message", "No error message returned"))

NameError: name 'ee' is not defined

In [ ]:
YEAR_RETRY = 1999
MONTH_RETRY = 4
RETRY_LABEL = "retry1"

retry_key = month_key(YEAR_RETRY, MONTH_RETRY)
retry_filename = (
    f"imerg_v07_hh_17huc02_{YEAR_RETRY:04d}_"
    f"{MONTH_RETRY:02d}_{RETRY_LABEL}_fixedgrid.csv"
)
retry_prefix = retry_filename.removesuffix(".csv")

retry_collection = build_month_collection(
    YEAR_RETRY, MONTH_RETRY
)

retry_count = retry_collection.size().getInfo()
retry_expected = expected_images(YEAR_RETRY, MONTH_RETRY)

print("Month:", retry_key)
print("Permanent images:", retry_count)
print("Expected images:", retry_expected)

if retry_count != retry_expected:
    raise RuntimeError(
        f"Source check failed: {retry_count} permanent images; "
        f"expected {retry_expected}."
    )

retry_table = retry_collection.map(
    image_to_basin_rows
).flatten()

retry_task = ee.batch.Export.table.toDrive(
    collection=retry_table,
    description=retry_prefix,
    folder=EXPORT_FOLDER_NAME,
    fileNamePrefix=retry_prefix,
    fileFormat="CSV",
    selectors=[
        "huc2", "name", "timestamp_utc",
        "precipitation_mm_per_hour", "valid_pixel_count",
        "status", "product"
    ]
)

retry_task.start()

print("Retry task ID:", retry_task.id)
print("Retry task state:", retry_task.status()["state"])

NameError: name 'month_key' is not defined

IMERG 2.0 ________----------------------_____________________---------------------

In [4]:
# =====================================================================
# IMERG HALF-HOURLY PIPELINE  --  SETUP CELL
# Run this cell FIRST in every new session. It is safe to run again and again.
# It never deletes or overwrites any file.
# =====================================================================
import re, shutil, calendar
from pathlib import Path
import pandas as pd
import ee
from google.colab import drive

drive.mount("/content/drive")

try:
    ee.Initialize(project="ecohydro-project")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ecohydro-project")

# ---------------- fixed settings ----------------
HUC2_CODES = ["01", "02", "03", "04", "05", "07", "08", "09", "10",
              "11", "12", "13", "14", "15", "16", "17", "18"]

EXPORT_FOLDER_NAME = "ecohydro_imerg_hh_export"            # Earth Engine writes here
EXPORT_DIR  = Path("/content/drive/MyDrive") / EXPORT_FOLDER_NAME
PROJECT_DIR = Path("/content/drive/MyDrive/ecohydro_project/data/raw/imerg_half_hourly")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PROJECT_DIR / "imerg_hh_export_manifest.csv"   # log of submissions
VERIFIED_LOG  = PROJECT_DIR / "imerg_hh_verified_months.csv"   # months that passed the audit

PREFIX = "imerg_v07_hh_17huc02_"
FIRST_YEAR, LAST_YEAR, LAST_MONTH_OF_LAST_YEAR = 1998, 2025, 9
ACTIVE_STATES = {"READY", "RUNNING"}
FAILED_STATES = {"FAILED", "CANCELLED", "CANCEL_REQUESTED"}
MANIFEST_COLUMNS = ["month_key", "year", "month", "filename", "task_id", "state", "note"]
REQUIRED_COLUMNS = {"huc2", "name", "timestamp_utc", "precipitation_mm_per_hour",
                    "valid_pixel_count", "status", "product"}

# ---------------- Earth Engine objects (nothing is downloaded here) ----------------
basins = (ee.FeatureCollection("USGS/WBD/2017/HUC02")
          .filter(ee.Filter.inList("huc2", HUC2_CODES))
          .select(["huc2", "name"]))
imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_V07").select("precipitation")
native_projection = (ee.Image(imerg.filterDate("2020-01-01", "2020-01-02").first())
                     .select("precipitation").projection())
reducer = ee.Reducer.mean().combine(reducer2=ee.Reducer.count(), sharedInputs=True)
assert basins.size().getInfo() == 17, "Expected 17 basins"

# ---------------- naming helpers ----------------
def months_to_process():
    return [(y, m) for y in range(FIRST_YEAR, LAST_YEAR + 1)
            for m in range(1, (LAST_MONTH_OF_LAST_YEAR if y == LAST_YEAR else 12) + 1)]

def month_key(y, m):
    return f"{y:04d}-{m:02d}"

def standard_filename(y, m):
    return f"{PREFIX}{y:04d}_{m:02d}_fixedgrid.csv"

def expected_images(y, m):
    return 48 * calendar.monthrange(y, m)[1]

def candidate_files(y, m):
    """Every CSV for this month in the project folder or the Earth Engine export folder."""
    found = []
    for folder in (PROJECT_DIR, EXPORT_DIR):
        if folder.is_dir():
            found += sorted(folder.glob(f"{PREFIX}{y:04d}_{m:02d}*.csv"))
    std = standard_filename(y, m)
    found.sort(key=lambda p: (p.name != std, str(p)))
    return found

# ---------------- the extraction itself (the only part that talks to Earth Engine data) ----------------
def build_month_collection(y, m):
    start = ee.Date.fromYMD(y, m, 1)
    end = start.advance(1, "month")
    return imerg.filterDate(start, end).filter(ee.Filter.eq("status", "permanent"))

def image_to_basin_rows(image):
    image = ee.Image(image)
    timestamp = ee.Date(image.get("system:time_start"))
    reduced = image.reduceRegions(collection=basins, reducer=reducer,
                                  crs=native_projection, tileScale=4)
    return reduced.map(lambda f: ee.Feature(None, {
        "huc2": f.get("huc2"),
        "name": f.get("name"),
        "timestamp_utc": timestamp.format("YYYY-MM-dd HH:mm:ss"),
        "precipitation_mm_per_hour": f.get("mean"),
        "valid_pixel_count": f.get("count"),
        "status": image.get("status"),
        "product": "NASA/GPM_L3/IMERG_V07"}))

def build_month_table(y, m):
    return build_month_collection(y, m).map(image_to_basin_rows).flatten()

# ---------------- saved memory (lives on Drive, survives disconnects) ----------------
def load_manifest():
    if MANIFEST_PATH.is_file():
        return pd.read_csv(MANIFEST_PATH, dtype={"month_key": "string", "task_id": "string"})
    return pd.DataFrame(columns=MANIFEST_COLUMNS)

def log_submission(key, y, m, filename, task_id, state, note=""):
    mf = load_manifest()
    mf.loc[len(mf)] = [key, y, m, filename, task_id, state, note]
    mf.to_csv(MANIFEST_PATH, index=False)

def load_verified():
    if VERIFIED_LOG.is_file():
        return set(pd.read_csv(VERIFIED_LOG, dtype=str)["month"].dropna())
    return set()

def save_verified(verified):
    pd.DataFrame({"month": sorted(verified)}).to_csv(VERIFIED_LOG, index=False)

# ---------------- what is Earth Engine doing right now? ----------------
_TASK_RE = re.compile(rf"^{PREFIX}(\d{{4}})_(\d{{2}})")

def get_task_table():
    rows = []
    for t in ee.batch.Task.list():
        s = t.status()
        match = _TASK_RE.match(s.get("description", ""))
        if not match:
            continue
        rows.append({"task_id": t.id, "description": s.get("description", ""),
                     "year": int(match.group(1)), "month": int(match.group(2)),
                     "state": s.get("state", ""),
                     "created_ms": s.get("creation_timestamp_ms", 0),
                     "eecu_seconds": s.get("batch_eecu_usage_seconds"),
                     "error": s.get("error_message", "")})
    cols = ["task_id", "description", "year", "month", "state",
            "created_ms", "eecu_seconds", "error"]
    return pd.DataFrame(rows, columns=cols)

def month_table(tasks=None):
    """One row per month (333 rows) with a simple state word."""
    if tasks is None:
        tasks = get_task_table()
    latest = {}
    for (y, m), g in tasks.groupby(["year", "month"]):
        latest[(y, m)] = g.sort_values("created_ms").iloc[-1]
    verified = load_verified()
    mf = load_manifest()
    source_failed = set(mf.loc[mf["state"] == "SOURCE_CHECK_FAILED", "month_key"].astype(str))
    rows = []
    for y, m in months_to_process():
        key = month_key(y, m)
        files = candidate_files(y, m)
        t = latest.get((y, m))
        if key in verified and (PROJECT_DIR / standard_filename(y, m)).is_file():
            state = "VERIFIED"
        elif key in verified:
            state = "VERIFIED_FILE_MISSING"
        elif files:
            state = "FILE_TO_AUDIT"
        elif t is not None:
            state = "COMPLETED_FILE_NOT_VISIBLE" if t["state"] == "COMPLETED" else t["state"]
        elif key in source_failed:
            state = "SOURCE_CHECK_FAILED"
        else:
            state = "NOT_SUBMITTED"
        rows.append({"month": key, "year": y, "monthnum": m, "state": state,
                     "task_id": t["task_id"] if t is not None else "",
                     "error": t["error"] if t is not None else ""})
    return pd.DataFrame(rows)

def show_status():
    tasks = get_task_table()
    mt = month_table(tasks)
    print(f"MONTH STATUS  (total months needed: {len(mt)})")
    print(mt["state"].value_counts().to_string())
    for st in ["FAILED", "CANCELLED", "COMPLETED_FILE_NOT_VISIBLE", "VERIFIED_FILE_MISSING",
               "SOURCE_CHECK_FAILED", "FILE_TO_AUDIT", "RUNNING", "READY"]:
        sub = mt[mt["state"] == st]
        if len(sub):
            print(f"\n{st}: " + ", ".join(sub["month"]))
            if st in FAILED_STATES:
                for r in sub.itertuples():
                    print(f"   {r.month}: {str(r.error)[:200]}")
    done = tasks[(tasks["state"] == "COMPLETED") & tasks["eecu_seconds"].notna()]
    if len(done):
        mean_s = done["eecu_seconds"].astype(float).mean()
        print(f"\nCOMPUTE USED by finished tasks: {done['eecu_seconds'].astype(float).sum()/3600:,.0f} EECU-hours "
              f"({len(done)} tasks, about {mean_s/3600:,.1f} EECU-hours per month)")
    n_ver = int((mt["state"] == "VERIFIED").sum())
    print(f"\nVERIFIED: {n_ver} / {len(mt)}")
    return mt

# ---------------- audit (checks every CSV before it is trusted) ----------------
def audit_month_file(path, y, m):
    df = pd.read_csv(path, dtype={"huc2": "string"})
    missing_cols = REQUIRED_COLUMNS - set(df.columns)
    if missing_cols:
        return [f"missing columns: {sorted(missing_cols)}"]
    ts = pd.to_datetime(df["timestamp_utc"], errors="coerce", utc=True)
    rain = pd.to_numeric(df["precipitation_mm_per_hour"], errors="coerce")
    pix = pd.to_numeric(df["valid_pixel_count"], errors="coerce")
    expected_times = pd.date_range(start=pd.Timestamp(year=y, month=m, day=1, tz="UTC"),
                                   periods=expected_images(y, m), freq="30min")
    issues = []
    if len(df) != len(HUC2_CODES) * len(expected_times):
        issues.append(f"rows={len(df)}; expected={len(HUC2_CODES) * len(expected_times)}")
    if ts.isna().any():
        issues.append("unparseable timestamps")
    if df.assign(parsed_time=ts).duplicated(["huc2", "parsed_time"]).any():
        issues.append("duplicate basin-timestamp rows")
    if set(df["huc2"].dropna()) != set(HUC2_CODES):
        issues.append("basin IDs do not match the 17 expected IDs")
    for code in HUC2_CODES:
        actual = pd.DatetimeIndex(ts.loc[df["huc2"] == code].dropna())
        miss, extra = expected_times.difference(actual), actual.difference(expected_times)
        if len(miss) or len(extra):
            issues.append(f"{code}: {len(miss)} missing, {len(extra)} extra timestamps")
    if rain.isna().any():
        issues.append(f"{int(rain.isna().sum())} missing rainfall values")
    if (rain < 0).any():
        issues.append(f"{int((rain < 0).sum())} negative rainfall values")
    if pix.isna().any() or (pix <= 0).any():
        issues.append("missing or nonpositive valid-pixel counts")
    if not df["status"].eq("permanent").all():
        issues.append("non-permanent images present")
    if not df["product"].eq("NASA/GPM_L3/IMERG_V07").all():
        issues.append("unexpected product identifier")
    return issues

def audit_new_exports():
    mt = month_table()
    verified = load_verified()
    new = []
    for r in mt[mt["state"] == "FILE_TO_AUDIT"].itertuples():
        y, m, key = r.year, r.monthnum, r.month
        std_path = PROJECT_DIR / standard_filename(y, m)
        good, notes = None, []
        for cand in candidate_files(y, m):
            try:
                issues = audit_month_file(cand, y, m)
            except Exception as exc:
                issues = [f"{type(exc).__name__}: {exc}"]
            if not issues:
                good = cand
                break
            notes.append(f"{cand.name}: " + "; ".join(issues))
        if good is None:
            print(key, "CHECK_NEEDED ->", " | ".join(notes))
            continue
        if good != std_path:
            if std_path.exists():
                print(key, "A valid file exists but another file already uses the standard name. "
                           "Please look at this month by hand:", good.name)
                continue
            shutil.copy2(good, std_path)
        verified.add(key)
        new.append(key)
        print(key, "VERIFIED")
    save_verified(verified)
    print(f"\nNewly verified: {len(new)}  |  Total verified: {len(verified)} / {len(mt)}")
    return sorted(verified)

# ---------------- submitting tasks ----------------
def _submit_one(y, m, note=""):
    key, fname = month_key(y, m), standard_filename(y, m)
    count = build_month_collection(y, m).size().getInfo()
    expected = expected_images(y, m)
    if count != expected:
        print(key, f"NOT SUBMITTED: found {count} permanent images, expected {expected}")
        log_submission(key, y, m, fname, "", "SOURCE_CHECK_FAILED",
                       f"permanent images={count}; expected={expected}")
        return False
    prefix = fname.removesuffix(".csv")
    task = ee.batch.Export.table.toDrive(
        collection=build_month_table(y, m), description=prefix,
        folder=EXPORT_FOLDER_NAME, fileNamePrefix=prefix, fileFormat="CSV",
        selectors=["huc2", "name", "timestamp_utc", "precipitation_mm_per_hour",
                   "valid_pixel_count", "status", "product"])
    task.start()
    log_submission(key, y, m, fname, task.id, task.status()["state"], note)
    print(key, "submitted:", task.id)
    return True

def submit_months(max_active=8, max_new=8, dry_run=False):
    mt = month_table()
    active = int(mt["state"].isin(ACTIVE_STATES).sum())
    todo = mt[mt["state"] == "NOT_SUBMITTED"]
    print(f"Tasks active now: {active} | months never submitted: {len(todo)}")
    submitted = 0
    for r in todo.itertuples():
        if active >= max_active or submitted >= max_new:
            break
        if dry_run:
            print("WOULD SUBMIT:", r.month)
        elif _submit_one(r.year, r.monthnum):
            active += 1
        submitted += 1
    print(f"Done. Handled this pass: {submitted}")

def retry_failed(confirm=False, max_new=4):
    mt = month_table()
    bad = mt[mt["state"].isin(FAILED_STATES)]
    if bad.empty:
        print("No failed months.")
        return
    print(bad[["month", "state", "error"]].to_string(index=False))
    if not confirm:
        print("\nNothing was resubmitted. To resubmit these, run: retry_failed(confirm=True)")
        return
    for r in bad.head(max_new).itertuples():
        _submit_one(r.year, r.monthnum, note=f"retry of failed task {r.task_id}")

print("SETUP COMPLETE. Next: run the STATUS cell.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETUP COMPLETE. Next: run the STATUS cell.


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [5]:
mt = show_status()

MONTH STATUS  (total months needed: 333)
state
NOT_SUBMITTED    280
FILE_TO_AUDIT     53

FILE_TO_AUDIT: 1998-01, 1998-02, 1998-03, 1998-04, 1998-05, 1998-06, 1998-07, 1998-08, 1998-09, 1998-10, 1998-11, 1998-12, 1999-01, 1999-02, 1999-03, 1999-04, 1999-05, 1999-06, 1999-07, 1999-08, 1999-09, 1999-10, 1999-11, 1999-12, 2000-01, 2000-02, 2000-03, 2000-04, 2000-05, 2000-06, 2000-07, 2000-08, 2000-09, 2000-10, 2000-11, 2000-12, 2001-01, 2001-02, 2001-03, 2001-04, 2001-05, 2001-06, 2001-07, 2001-08, 2001-09, 2001-10, 2001-11, 2001-12, 2002-01, 2002-02, 2002-03, 2002-04, 2020-01

COMPUTE USED by finished tasks: 1,308 EECU-hours (31 tasks, about 42.2 EECU-hours per month)

VERIFIED: 0 / 333


In [6]:
verified_months = audit_new_exports()

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


1998-01 VERIFIED
1998-02 VERIFIED
1998-03 VERIFIED
1998-04 VERIFIED
1998-05 VERIFIED
1998-06 VERIFIED
1998-07 VERIFIED
1998-08 VERIFIED
1998-09 VERIFIED
1998-10 VERIFIED
1998-11 VERIFIED
1998-12 VERIFIED
1999-01 VERIFIED
1999-02 VERIFIED
1999-03 VERIFIED
1999-04 VERIFIED
1999-05 VERIFIED
1999-06 VERIFIED
1999-07 VERIFIED
1999-08 VERIFIED
1999-09 VERIFIED
1999-10 VERIFIED
1999-11 VERIFIED
1999-12 VERIFIED
2000-01 VERIFIED
2000-02 VERIFIED
2000-03 VERIFIED
2000-04 VERIFIED
2000-05 VERIFIED
2000-06 VERIFIED
2000-07 VERIFIED
2000-08 VERIFIED
2000-09 VERIFIED
2000-10 VERIFIED
2000-11 VERIFIED
2000-12 VERIFIED
2001-01 VERIFIED
2001-02 VERIFIED
2001-03 VERIFIED
2001-04 VERIFIED
2001-05 VERIFIED
2001-06 VERIFIED
2001-07 VERIFIED
2001-08 VERIFIED
2001-09 VERIFIED
2001-10 VERIFIED
2001-11 VERIFIED
2001-12 VERIFIED
2002-01 VERIFIED
2002-02 VERIFIED
2002-03 VERIFIED
2002-04 VERIFIED
2020-01 VERIFIED

Newly verified: 53  |  Total verified: 53 / 333


In [7]:
submit_months(max_active=8, max_new=8, dry_run=True)

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Tasks active now: 0 | months never submitted: 280
WOULD SUBMIT: 2002-05
WOULD SUBMIT: 2002-06
WOULD SUBMIT: 2002-07
WOULD SUBMIT: 2002-08
WOULD SUBMIT: 2002-09
WOULD SUBMIT: 2002-10
WOULD SUBMIT: 2002-11
WOULD SUBMIT: 2002-12
Done. Handled this pass: 8


In [8]:
submit_months(max_active=8, max_new=8)

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Tasks active now: 0 | months never submitted: 280


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


2002-05 submitted: W2SIDSVM6YN6526VBYFWATKW


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


...

2002-10 submitted: SXPPT2WWRTHN46AV6VHKXEIJ


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


2002-11 submitted: D7Z4N3ICZB34BYDJEVHDKKY7


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


2002-12 submitted: Q7LCJBHDRJ2WZHQC7ZCX4U4G
Done. Handled this pass: 8


In [9]:
retry_failed()

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


No failed months.


In [10]:
retry_failed(confirm=True)

/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


No failed months.


###ERA5 single level hourly data-----------------------------------------------------------------------------


In [ ]:
# ============================================================
# ERA5 inspection - Part 1
# Folder and file structure
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
import os
import pandas as pd

BASE_DIR = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/ERA5_Single_Levels"
)

if not BASE_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {BASE_DIR}")

print(f"Base folder: {BASE_DIR}")
print(f"Exists: {BASE_DIR.exists()}")
print()

all_paths = sorted(BASE_DIR.rglob("*"))

files = [p for p in all_paths if p.is_file()]
folders = [p for p in all_paths if p.is_dir()]

print(f"Number of folders: {len(folders)}")
print(f"Number of files: {len(files)}")
print()

# File extension summary
extension_counts = Counter()

for path in files:
    suffix = path.suffix.lower()
    if suffix == "":
        suffix = "[no extension]"
    extension_counts[suffix] += 1

print("File types:")
for suffix, count in sorted(extension_counts.items()):
    print(f"  {suffix}: {count}")

print()
print("Total storage used:")

total_bytes = sum(path.stat().st_size for path in files)
print(f"  {total_bytes / 1024**3:.2f} GB")

print()
print("Folder tree:")
print("-" * 100)

for path in all_paths:
    relative = path.relative_to(BASE_DIR)
    depth = len(relative.parts) - 1
    indent = "    " * depth

    if path.is_dir():
        print(f"{indent}[DIR]  {path.name}/")
    else:
        size_mb = path.stat().st_size / 1024**2
        print(f"{indent}[FILE] {path.name}  ({size_mb:.2f} MB)")

print("-" * 100)

# Save a file inventory for later inspection
inventory = []

for path in files:
    stat = path.stat()
    inventory.append({
        "relative_path": str(path.relative_to(BASE_DIR)),
        "absolute_path": str(path),
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": stat.st_size,
        "size_mb": stat.st_size / 1024**2,
        "parent_folder": path.parent.name
    })

inventory_df = pd.DataFrame(inventory)

inventory_file = BASE_DIR.parent / "ERA5_file_inventory.csv"
inventory_df.to_csv(inventory_file, index=False)

print()
print(f"File inventory saved to:")
print(inventory_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base folder: /content/drive/MyDrive/ecohydro_project/data/raw/ERA5_Single_Levels
Exists: True

Number of folders: 20
Number of files: 15

File types:
  .nc: 15

Total storage used:
  42.51 GB

Folder tree:
----------------------------------------------------------------------------------------------------
[DIR]  mvimd/
    [DIR]  4e6b894fdf5ff722b1ac4518ad1fdc84/
        [FILE] data_stream-oper_stepType-avg.nc  (3855.07 MB)
    [DIR]  a909743a133accffdc177a1e9148799f/
        [FILE] data_stream-oper_stepType-avg.nc  (2002.56 MB)
    [DIR]  e7cfd862f9617ed0677b75a4b7349db2/
        [FILE] data_stream-oper_stepType-avg.nc  (3794.02 MB)
[DIR]  vidmf/
    [DIR]  1bab39f7ccea107dbde1a509e1081ced/
        [FILE] data_stream-oper_stepType-instant.nc  (801.62 MB)
    [DIR]  594addfcb0b18e56f47c6158777ab339/
        [FILE] data_stream-oper_stepType-instant.nc  (3871.0

In [ ]:
# ============================================================
# ERA5 Stage 3.10 - complete file inspection
# Metadata only; does not load full data arrays
# ============================================================

from pathlib import Path
from collections import defaultdict
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")

BASE_DIR = Path(
    "/content/drive/MyDrive/ecohydro_project/data/raw/ERA5_Single_Levels"
)

REPORT_DIR = BASE_DIR.parent / "ERA5_inspection_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Report directory: {REPORT_DIR}")

# ------------------------------------------------------------
# File discovery
# ------------------------------------------------------------

DATA_EXTENSIONS = {
    ".nc", ".nc4", ".netcdf",
    ".grib", ".grb", ".grib1", ".grib2", ".grb2"
}

data_files = sorted([
    path for path in BASE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in DATA_EXTENSIONS
])

print(f"Data files found: {len(data_files)}")

if len(data_files) == 0:
    raise FileNotFoundError("No NetCDF or GRIB files were found.")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def open_data_file(path):
    suffix = path.suffix.lower()

    if suffix in {".grib", ".grb", ".grib1", ".grib2", ".grb2"}:
        return xr.open_dataset(
            path,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""}
        )

    return xr.open_dataset(
        path,
        decode_times=True,
        mask_and_scale=False
    )

def find_coordinate(ds, possible_names):
    available = list(ds.coords) + list(ds.variables)

    for name in available:
        if name.lower() in possible_names:
            return name

    return None

def find_time_coordinate(ds):
    return find_coordinate(
        ds,
        {
            "time",
            "valid_time",
            "forecast_time",
            "date",
            "datetime"
        }
    )

def safe_datetime_index(values):
    try:
        result = pd.to_datetime(values)

        if getattr(result, "tz", None) is not None:
            result = result.tz_localize(None)

        return pd.DatetimeIndex(result)

    except Exception:
        return pd.DatetimeIndex([])

def coordinate_summary(ds, coordinate_name):
    if coordinate_name is None:
        return {
            "count": np.nan,
            "minimum": np.nan,
            "maximum": np.nan,
            "resolution": np.nan,
            "ascending": None
        }

    values = np.asarray(ds[coordinate_name].values).astype(float)
    values = values.ravel()

    if len(values) == 0:
        return {
            "count": 0,
            "minimum": np.nan,
            "maximum": np.nan,
            "resolution": np.nan,
            "ascending": None
        }

    if len(values) > 1:
        differences = np.diff(values)
        resolution = float(np.nanmedian(np.abs(differences)))
        ascending = bool(np.all(differences > 0))
    else:
        resolution = np.nan
        ascending = None

    return {
        "count": len(values),
        "minimum": float(np.nanmin(values)),
        "maximum": float(np.nanmax(values)),
        "resolution": resolution,
        "ascending": ascending
    }

def clean_value(value):
    if value is None:
        return ""

    if isinstance(value, np.ndarray):
        return str(value.tolist())

    return str(value)

# ------------------------------------------------------------
# Inspect all files
# ------------------------------------------------------------

file_records = []
variable_records = []
time_records = []
file_times = {}

for i, path in enumerate(data_files, start=1):

    relative_path = str(path.relative_to(BASE_DIR))
    folder_label = path.relative_to(BASE_DIR).parts[0]

    print(f"[{i}/{len(data_files)}] {relative_path}")

    file_record = {
        "relative_path": relative_path,
        "folder_label": folder_label,
        "filename": path.name,
        "size_gb": path.stat().st_size / 1024**3,
        "status": "",
        "error": ""
    }

    try:
        ds = open_data_file(path)

        file_record["status"] = "OPENED"
        file_record["data_variables"] = "; ".join(ds.data_vars)
        file_record["coordinate_variables"] = "; ".join(ds.coords)
        file_record["dimensions"] = json.dumps({
            key: int(value) for key, value in ds.sizes.items()
        })

        time_name = find_time_coordinate(ds)
        lat_name = find_coordinate(ds, {"latitude", "lat", "y"})
        lon_name = find_coordinate(ds, {"longitude", "lon", "x"})

        file_record["time_coordinate"] = time_name or ""
        file_record["latitude_coordinate"] = lat_name or ""
        file_record["longitude_coordinate"] = lon_name or ""

        if time_name is not None:
            times = safe_datetime_index(ds[time_name].values)
            file_times[relative_path] = times

            file_record["time_count"] = len(times)
            file_record["time_start"] = (
                str(times.min()) if len(times) else ""
            )
            file_record["time_end"] = (
                str(times.max()) if len(times) else ""
            )

            if len(times) > 1:
                sorted_times = pd.DatetimeIndex(sorted(times))
                diffs = pd.Series(sorted_times).diff().dropna()

                file_record["median_time_step"] = str(diffs.median())
                file_record["minimum_time_step"] = str(diffs.min())
                file_record["maximum_time_step"] = str(diffs.max())
                file_record["duplicate_times_in_file"] = int(
                    times.duplicated().sum()
                )

                file_record["hours_present"] = ",".join(
                    map(str, sorted(set(times.hour)))
                )
                file_record["minutes_present"] = ",".join(
                    map(str, sorted(set(times.minute)))
                )

        else:
            file_record["time_count"] = np.nan
            file_record["time_start"] = ""
            file_record["time_end"] = ""

        lat_info = coordinate_summary(ds, lat_name)
        lon_info = coordinate_summary(ds, lon_name)

        file_record["latitude_count"] = lat_info["count"]
        file_record["latitude_min"] = lat_info["minimum"]
        file_record["latitude_max"] = lat_info["maximum"]
        file_record["latitude_resolution"] = lat_info["resolution"]
        file_record["latitude_ascending"] = lat_info["ascending"]

        file_record["longitude_count"] = lon_info["count"]
        file_record["longitude_min"] = lon_info["minimum"]
        file_record["longitude_max"] = lon_info["maximum"]
        file_record["longitude_resolution"] = lon_info["resolution"]
        file_record["longitude_ascending"] = lon_info["ascending"]

        # Variable-level metadata
        for variable_name in ds.data_vars:

            da = ds[variable_name]

            variable_records.append({
                "relative_path": relative_path,
                "folder_label": folder_label,
                "variable_name": variable_name,
                "dimensions": str(da.dims),
                "shape": str(da.shape),
                "dtype": str(da.dtype),
                "units": clean_value(da.attrs.get("units")),
                "long_name": clean_value(da.attrs.get("long_name")),
                "standard_name": clean_value(
                    da.attrs.get("standard_name")
                ),
                "shortName": clean_value(
                    da.attrs.get("shortName")
                ),
                "paramId": clean_value(
                    da.attrs.get("paramId")
                ),
                "GRIB_cfName": clean_value(
                    da.attrs.get("GRIB_cfName")
                ),
                "GRIB_shortName": clean_value(
                    da.attrs.get("GRIB_shortName")
                ),
                "GRIB_paramId": clean_value(
                    da.attrs.get("GRIB_paramId")
                ),
                "coordinates": clean_value(
                    da.attrs.get("coordinates")
                )
            })

        # Detailed time record
        if time_name is not None:
            times = file_times[relative_path]

            if len(times) > 0:

                sorted_times = pd.DatetimeIndex(sorted(times))
                expected_hours = {0, 6, 12, 18}
                observed_hours = set(sorted_times.hour)

                expected_sequence = pd.date_range(
                    start=sorted_times.min(),
                    end=sorted_times.max(),
                    freq="6h"
                )

                missing_times = expected_sequence.difference(
                    sorted_times.drop_duplicates()
                )

                wrong_hours = sorted(
                    observed_hours - expected_hours
                )

                time_records.append({
                    "relative_path": relative_path,
                    "folder_label": folder_label,
                    "time_count": len(times),
                    "unique_time_count": len(times.unique()),
                    "duplicate_time_count": int(
                        times.duplicated().sum()
                    ),
                    "time_start": times.min(),
                    "time_end": times.max(),
                    "hours_present": ",".join(
                        map(str, sorted(observed_hours))
                    ),
                    "minutes_present": ",".join(
                        map(str, sorted(set(sorted_times.minute)))
                    ),
                    "wrong_hours": ",".join(
                        map(str, wrong_hours)
                    ),
                    "expected_6hourly_count": len(expected_sequence),
                    "missing_6hourly_count": len(missing_times),
                    "median_step": str(
                        pd.Series(sorted_times).diff().dropna().median()
                    ),
                    "minimum_step": str(
                        pd.Series(sorted_times).diff().dropna().min()
                    ),
                    "maximum_step": str(
                        pd.Series(sorted_times).diff().dropna().max()
                    )
                })

        ds.close()

    except Exception as exc:

        file_record["status"] = "ERROR"
        file_record["error"] = f"{type(exc).__name__}: {exc}"

    file_records.append(file_record)

# ------------------------------------------------------------
# Save basic reports
# ------------------------------------------------------------

files_df = pd.DataFrame(file_records)
variables_df = pd.DataFrame(variable_records)
times_df = pd.DataFrame(time_records)

files_df.to_csv(
    REPORT_DIR / "file_metadata_summary.csv",
    index=False
)

variables_df.to_csv(
    REPORT_DIR / "variable_metadata_summary.csv",
    index=False
)

times_df.to_csv(
    REPORT_DIR / "file_time_summary.csv",
    index=False
)

# ------------------------------------------------------------
# Identify actual variable-folder mismatches
# ------------------------------------------------------------

def normalise_text(value):
    return (
        str(value)
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )

variable_mismatch_records = []

for _, row in variables_df.iterrows():

    folder = normalise_text(row["folder_label"])
    variable_name = normalise_text(row["variable_name"])
    short_name = normalise_text(row["shortName"])
    grib_short_name = normalise_text(row["GRIB_shortName"])

    likely_match = (
        folder in variable_name
        or folder in short_name
        or folder in grib_short_name
    )

    variable_mismatch_records.append({
        "relative_path": row["relative_path"],
        "folder_label": row["folder_label"],
        "actual_variable_name": row["variable_name"],
        "shortName": row["shortName"],
        "GRIB_shortName": row["GRIB_shortName"],
        "units": row["units"],
        "long_name": row["long_name"],
        "likely_folder_match": likely_match
    })

mismatch_df = pd.DataFrame(variable_mismatch_records)

mismatch_df.to_csv(
    REPORT_DIR / "folder_vs_actual_variable_check.csv",
    index=False
)

# ------------------------------------------------------------
# Group files by actual variable
# ------------------------------------------------------------

actual_variable_groups = defaultdict(list)

for _, row in variables_df.iterrows():
    actual_variable_groups[row["variable_name"]].append(
        row["relative_path"]
    )

group_records = []

for variable_name, paths in actual_variable_groups.items():

    group_times = []

    for relative_path in paths:
        if relative_path in file_times:
            group_times.extend(file_times[relative_path].tolist())

    if len(group_times) == 0:
        continue

    group_times = pd.DatetimeIndex(sorted(group_times))
    unique_times = group_times.drop_duplicates()

    expected_sequence = pd.date_range(
        start=unique_times.min(),
        end=unique_times.max(),
        freq="6h"
    )

    missing_times = expected_sequence.difference(unique_times)

    group_records.append({
        "actual_variable_name": variable_name,
        "number_of_files": len(paths),
        "total_time_values_including_overlap": len(group_times),
        "unique_time_values": len(unique_times),
        "duplicate_time_values_across_files": int(
            group_times.duplicated().sum()
        ),
        "group_start": unique_times.min(),
        "group_end": unique_times.max(),
        "hours_present": ",".join(
            map(str, sorted(set(unique_times.hour)))
        ),
        "minutes_present": ",".join(
            map(str, sorted(set(unique_times.minute)))
        ),
        "expected_6hourly_values": len(expected_sequence),
        "missing_6hourly_values": len(missing_times),
        "files": "; ".join(paths)
    })

groups_df = pd.DataFrame(group_records)

groups_df.to_csv(
    REPORT_DIR / "actual_variable_group_summary.csv",
    index=False
)

# ------------------------------------------------------------
# Detect file-period overlaps
# ------------------------------------------------------------

file_periods = files_df[
    ["relative_path", "folder_label", "time_start", "time_end"]
].copy()

file_periods["time_start"] = pd.to_datetime(
    file_periods["time_start"], errors="coerce"
)

file_periods["time_end"] = pd.to_datetime(
    file_periods["time_end"], errors="coerce"
)

overlap_records = []

for folder, folder_df in file_periods.groupby("folder_label"):

    folder_df = folder_df.dropna(
        subset=["time_start", "time_end"]
    ).sort_values("time_start").reset_index(drop=True)

    for i in range(len(folder_df)):

        for j in range(i + 1, len(folder_df)):

            a = folder_df.iloc[i]
            b = folder_df.iloc[j]

            overlap_start = max(a["time_start"], b["time_start"])
            overlap_end = min(a["time_end"], b["time_end"])

            if overlap_start <= overlap_end:

                overlap_records.append({
                    "folder_label": folder,
                    "file_a": a["relative_path"],
                    "file_b": b["relative_path"],
                    "file_a_start": a["time_start"],
                    "file_a_end": a["time_end"],
                    "file_b_start": b["time_start"],
                    "file_b_end": b["time_end"],
                    "overlap_start": overlap_start,
                    "overlap_end": overlap_end,
                    "overlap_days": (
                        overlap_end - overlap_start
                    ).total_seconds() / 86400
                })

overlaps_df = pd.DataFrame(overlap_records)

overlaps_df.to_csv(
    REPORT_DIR / "overlapping_file_periods.csv",
    index=False
)

# ------------------------------------------------------------
# Exact duplicate file hashes
# ------------------------------------------------------------

def md5_hash(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

hash_records = []

for path in data_files:

    hash_records.append({
        "relative_path": str(path.relative_to(BASE_DIR)),
        "size_bytes": path.stat().st_size,
        "md5": md5_hash(path)
    })

hash_df = pd.DataFrame(hash_records)

exact_duplicates_df = (
    hash_df.groupby("md5")
    .filter(lambda group: len(group) > 1)
)

hash_df.to_csv(
    REPORT_DIR / "file_hashes.csv",
    index=False
)

exact_duplicates_df.to_csv(
    REPORT_DIR / "exact_duplicate_files.csv",
    index=False
)

# ------------------------------------------------------------
# Print the main results
# ------------------------------------------------------------

print()
print("=" * 100)
print("1. FILE OPENING STATUS")
print("=" * 100)
print(files_df["status"].value_counts(dropna=False))

print()
print("=" * 100)
print("2. ACTUAL VARIABLES FOUND")
print("=" * 100)
print(
    variables_df[
        [
            "relative_path",
            "folder_label",
            "variable_name",
            "shortName",
            "GRIB_shortName",
            "paramId",
            "units",
            "long_name"
        ]
    ].to_string(index=False)
)

print()
print("=" * 100)
print("3. FOLDER LABEL VS ACTUAL VARIABLE")
print("=" * 100)
print(mismatch_df.to_string(index=False))

print()
print("=" * 100)
print("4. ACTUAL VARIABLE GROUP SUMMARY")
print("=" * 100)
print(groups_df.to_string(index=False))

print()
print("=" * 100)
print("5. FILE PERIOD OVERLAPS")
print("=" * 100)

if overlaps_df.empty:
    print("No overlapping file periods detected.")
else:
    print(overlaps_df.to_string(index=False))

print()
print("=" * 100)
print("6. EXACT DUPLICATE FILES")
print("=" * 100)

if exact_duplicates_df.empty:
    print("No exact duplicate files detected.")
else:
    print(exact_duplicates_df.to_string(index=False))

print()
print("=" * 100)
print("REPORT FILES")
print("=" * 100)

for path in sorted(REPORT_DIR.glob("*.csv")):
    print(path)

Base directory: /content/drive/MyDrive/ecohydro_project/data/raw/ERA5_Single_Levels
Report directory: /content/drive/MyDrive/ecohydro_project/data/raw/ERA5_inspection_reports
Data files found: 15
[1/15] mvimd/4e6b894fdf5ff722b1ac4518ad1fdc84/data_stream-oper_stepType-avg.nc
[2/15] mvimd/a909743a133accffdc177a1e9148799f/data_stream-oper_stepType-avg.nc
[3/15] mvimd/e7cfd862f9617ed0677b75a4b7349db2/data_stream-oper_stepType-avg.nc
[4/15] vidmf/1bab39f7ccea107dbde1a509e1081ced/data_stream-oper_stepType-instant.nc
[5/15] vidmf/594addfcb0b18e56f47c6158777ab339/data_stream-oper_stepType-instant.nc
[6/15] vidmf/9c45952d5872fc5c920e45589a2246c0/data_stream-oper_stepType-instant.nc
[7/15] vimd/4886ade09d0004f3faae6c47fc957827/data_stream-oper_stepType-accum.nc
[8/15] vimd/e6160d19e3017a52d92bc434d1daa99/data_stream-oper_stepType-accum.nc
[9/15] vimd/f7bd38a7f726e82384f4046847b5da6d/data_stream-oper_stepType-accum.nc
[10/15] viwve/89a7a284c7975d28ba74f04567df3b8b/data_stream-oper_stepType-instan

###ERA 5 Soil moisture gridded data---------------------------------------------------------